# Movie Opening-Weekend Data Pipeline: TMDB Ingestion, Box Office Scraping, Star Schema & Feature Engineering

## Table of Contents

### Introduction & Background
- [Overview](#Overview)
- [The Problem](#The-Problem)
- [What This Pipeline Does](#What-This-Pipeline-Does)
  - [Pipeline Flow](#Pipeline-Flow)

### Architecture
- [Architecture: Three-Layer Design](#Architecture:-Three-Layer-Design)
  - [Layer 1: Staging (Raw Ingestion)](#Layer-1:-Staging-(Raw-Ingestion))
  - [Layer 2: Warehouse (Star Schema)](#Layer-2:-Warehouse-(Star-Schema))
  - [Layer 3: Analytics (Final Products)](#Layer-3:-Analytics-(Final-Products))
- [Pipeline Architecture Diagram](#Pipeline-Architecture-Diagram)
- [Why This Approach?](#Why-This-Approach?)
- [Data Sources](#Data-Sources)

### Feature Documentation
- [Feature Engineering](#Feature-Engineering)
- [Feature Rationale](#Feature-Rationale)
- [Complete Feature Table](#Complete-Feature-Table)

### Design & Validation
- [Key Design Decisions](#Key-Design-Decisions)
- [Data Validation](#Data-Validation)

### Usage Guide
- [Quick Start](#Quick-Start)
- [Output Files](#Output-Files)
- [Notebook Sections](#Notebook-Sections)

### Implementation
- [Section 0 - Imports and Global Config](#Section-0---Imports-and-Global-Config)
- [Section 1 - TMDB Helpers and Utilities](#Section-1---TMDB-Helpers-and-Utilities)
- [Section 2 - HTML Caching & Box Office Parsing](#Section-2---HTML-Caching-&-Box-Office-Parsing)
- [Section 3 - Rating, Studio & Season Helpers](#Section-3---Rating,-Studio-&-Season-Helpers)
- [Section 4 - TMDB Stage](#Section-4---TMDB-Stage)
- [Section 5 - Opening Weekend Staging](#Section-5---Opening-Weekend-Staging)
- [Section 6 - Dimensions](#Section-6---Dimensions)
- [Section 7 - Fact: Opening Weekend](#Section-7---Fact:-Opening-Weekend)
- [Section 8 - Flat Raw Dataset](#Section-8---Flat-Raw-Dataset)
- [Section 9 - Raw Avatar 3 Only](#Section-9---Raw-Avatar-3-Only)
- [Section 10 - Feature Engineering Helpers](#Section-10---Feature-Engineering-Helpers)
- [Section 11 - Feature Engineering Pipeline](#Section-11---Feature-Engineering-Pipeline)
- [Section 12 - Main](#Section-12---Main)

## Overview

This notebook builds an end-to-end movie opening-weekend analytics dataset using a fully automated data-engineering pipeline. It integrates metadata from TMDB, opening-weekend revenue from Box Office Mojo and The Numbers, and organizes everything into a star schema before producing flattened and fully engineered machine-learning-ready feature tables.

All operations are parameterized, reproducible, and require no manual file uploads.

## The Problem

Opening weekend box office is the most critical metric in film economics, determining a movie's financial trajectory and theatrical run length. However, building a predictive dataset is non-trivial:

- Metadata lives in TMDB (budget, runtime, cast, genres)
- Revenue data is scattered across Box Office Mojo and The Numbers
- Each source uses different formats and has missing values
- Historical franchise data requires temporal lookups
- Studio classification requires fuzzy text matching across hundreds of production company name variations

Most approaches involve manual CSV downloads, Excel merges, and ad-hoc feature engineering. This pipeline automates the entire process and produces a reproducible, auditable dataset.

## What This Pipeline Does

**Input**: Year range (e.g., 2007–2025) and minimum vote threshold

**Output**: Star schema CSVs, flattened raw datasets, and ML-ready feature tables with 30+ engineered features

### Pipeline Flow

```
TMDB API (metadata)
        ↓
Box Office Mojo + The Numbers (scraping with fallback)
        ↓
Star Schema (dimensional modeling)
        ↓
Flattened datasets (denormalized)
        ↓
Feature engineering (temporal, franchise, studio, popularity)
        ↓
preprocessed_movies_all.csv (training data)
preprocessed_avatar3_only.csv (prediction target)
```

## Architecture: Three-Layer Design

This pipeline follows standard data warehouse architecture with clear separation between raw ingestion, modeled storage, and analytical products.

### Layer 1: Staging (Raw Ingestion)

Unmodeled, untransformed data exactly as received from sources:

- `stg_tmdb.csv` - Raw TMDB API responses (one row per movie)
- `stg_opening.csv` - Scraped opening weekend revenue with source attribution
- `cache_html/` - Cached HTML pages from Box Office Mojo and The Numbers

These files are never used directly for analysis. They exist for debugging, data lineage, and re-processing without hitting external APIs again.

### Layer 2: Warehouse (Star Schema)

Cleaned, normalized, relational tables following dimensional modeling best practices:

**Dimensions**:
- `dim_date.csv` - Calendar dimension (date_key, year, month, day, weekday)
- `dim_rating.csv` - MPA ratings (G, PG, PG-13, R, NC-17, etc.)
- `dim_movie.csv` - Movie master table with foreign keys to date and rating
- `dim_studio.csv` - Production companies with canonical names

**Bridge Table**:
- `bridge_movie_studio.csv` - Many-to-many relationship (movies often have multiple studios)

**Fact Table**:
- `fact_opening_weekend.csv` - The measurable business event (opening revenue, source, URL)

This structure enables complex analytical queries without JOIN hell and maintains referential integrity through surrogate keys.

### Layer 3: Analytics (Final Products)

Denormalized, ML-ready datasets:

**Raw Flat Outputs**:
- `raw_movies_all.csv` - All dimensions joined (human-readable, good for exploratory analysis)
- `raw_avatar3_only.csv` - Single-row snapshot of Avatar 3 metadata

**Engineered ML Feature Outputs**:
- `preprocessed_movies_all.csv` - Training dataset with 30+ engineered features (excludes Avatar 3)
- `preprocessed_avatar3_only.csv` - Same features, prediction target only (opening_weekend = NaN)

## Pipeline Architecture Diagram

```
┌──────────────────────────────┐
│         TMDB API             │
│  (metadata: cast, crew,      │
│   budget, genre, ratings)    │
└──────────────┬───────────────┘
               │
               ▼
        ┌──────────────────┐
        │    stg_tmdb      │
        │  (raw metadata)  │
        └───────┬──────────┘
                │
     ┌──────────┴──────────┐
     │                     │
     ▼                     ▼
┌──────────────────┐  ┌───────────────────┐
│ Box Office Mojo  │  │   The Numbers     │
│ (opening weekend)│  │ (fallback scrape) │
└──────────────────┘  └───────────────────┘
     │                     │
     └──────────┬──────────┘
                ▼
         ┌────────────────┐
         │  stg_opening   │
         │ (DOM opening $)│
         └──────┬─────────┘
                │
════════════════╪════════════════════════════════
  STAGING LAYER │ (raw ingestion)
════════════════╪════════════════════════════════
 WAREHOUSE LAYER│ (modeled star schema)
════════════════╪════════════════════════════════
                │
     ┌──────────┼──────────┐
     │          │          │
     ▼          ▼          ▼
┌────────────┐ ┌────────────┐ ┌────────────┐
│  dim_date  │ │ dim_rating │ │ dim_studio │
└──────┬─────┘ └──────┬─────┘ └──────┬─────┘
       │              │              │
       └──────┐ ┌─────┘              │
              ▼ ▼                    │
        ┌────────────┐               │
        │  dim_movie │◄──────────────┘
        │(per-movie) │◄─────────────┐
        └──────┬─────┘              │
               │              ┌─────────────────┐
               │              │ bridge_movie_   │
               │              │     studio      │
               │              └─────────────────┘
               ▼
      ┌──────────────────┐
      │  fact_opening_   │
      │     weekend      │
      └──────┬───────────┘
             │
═════════════╪══════════════════════════════════
  WAREHOUSE  │ (source of truth)
═════════════╪══════════════════════════════════
 ANALYTICAL  │ (final deliverables)
  PRODUCTS   │
═════════════╪══════════════════════════════════
             │
             ▼
    ┌────────────────────┐
    │  raw_movies_all    │
    │ denormalized data  │
    │       mart         │
    └─────────┬──────────┘
              │
       ┌──────┴──────┐
       ▼             ▼
┌──────────────┐ ┌──────────────────┐
│preprocessed_ │ │ preprocessed_    │
│movies_all.csv│ │avatar3_only.csv  │
│(ML-ready     │ │(ML-ready         │
│ features)    │ │ features)        │
└──────────────┘ └──────────────────┘
```

## Why This Approach?

### The Naive Alternative

1. Download movie CSV from Kaggle
2. Manually Google opening weekend revenue
3. Merge datasets in Excel with VLOOKUP
4. Throw raw columns into a regression model

**Problems**: Stale data, no reproducibility, no data lineage, manual errors, no validation.

### The Engineering Approach (This Notebook)

1. Pull fresh data from APIs
2. Build a proper dimensional model with foreign keys
3. Automate feature engineering with clear transformations
4. Validate data quality at every stage
5. Produce reproducible, testable outputs

**Benefits**: Reproducible, auditable, extensible, production-ready.

## Data Sources

| Source | Data | Coverage | Notes |
|--------|------|----------|-------|
| TMDB API | Metadata (budget, runtime, cast, crew, genres, ratings) | Configurable | Filtered by vote count (Configurable) |
| Box Office Mojo | Domestic opening weekend revenue | ~80% of movies | Scraped via IMDb ID |
| The Numbers | Domestic opening weekend revenue (fallback) | ~95% of movies | Scraped via title+year slug |

**Planned Configuration (for full dataset generation)**

* The final model will use movies released between **2007 and 2025**.
  The year **2007** is chosen as the starting point because it is widely regarded as the beginning of the modern **streaming era**, which significantly changed studio strategies, release patterns, and box-office dynamics.
* The per-year TMDB discovery limit will be 1000 per year to produce a dataset of approximately **10,000 movies**.
* Notebook runtime will scale **linearly** with dataset size due to rate-limited TMDB API requests and the cost of web scraping.


## Feature Engineering

The pipeline produces ~35-40 features across several categories (exact count depends on genres and whether recent-opening features are enabled):

### Temporal Features

Release window has a massive impact on box office. The pipeline encodes five mutually exclusive seasonal windows:

- `is_summer` (May 1–Aug 31): Summer blockbuster season
- `is_holiday` (Nov 15–Dec 31): Holiday releases (Thanksgiving, Christmas)
- `is_q1_dump_window` (Jan 1–Feb 20): January dump period for weak releases
- `is_spring_dead_zone` (Feb 21–Apr 30): Low box office period
- `is_awards_season` (Oct 1–Nov 14): Oscar bait releases

### Franchise Features

Sequels and franchises have established audiences. The pipeline looks backward in time:

- `is_collection`: Binary flag (is this part of a franchise?)
- `collection_past_open_avg`: Mean opening weekend of all previous films in this franchise (strictly prior releases only, no data leakage)

Example: Avatar 3 would have `collection_past_open_avg` = mean of Avatar 1 and Avatar 2 openings.

### Studio & Language Features

Major studios have distribution power and marketing budgets. The pipeline uses fuzzy text matching to classify studios:

- `is_major_studio`: Binary flag for Big 5 + Lionsgate + A24 (matches 40+ studio name variations)
- `is_english`: Binary flag (original_language == "en")

### Star Power Features

Director and lead actor track records, computed via TMDB filmography lookups:

**Always computed**:
- `director_has_prior`: Binary flag (director has prior feature films)
- `director_prior_count`: Number of prior films directed
- `lead_actor_has_prior`: Binary flag (lead actor has prior feature films)
- `lead_actor_prior_count`: Number of prior films for lead actor

**Conditional (only when `FETCH_RECENT_OPENING_FEATURES=True`)**:
- `director_recent_opening`: Most recent opening weekend from director's prior films
- `lead_actor_recent_opening`: Most recent opening weekend from lead actor's prior films

### Economic Features

- `log_budget`: Log-transformed budget using log1p (after studio+genre hierarchy imputation)
- `has_budget`: Binary flag (1 if TMDB had budget data before imputation, 0 if imputed)
- `runtime_minutes`: Film length (raw, no transformation)

### Categorical Encodings

- `rating_*`: One-hot encoding of normalized MPA ratings (rating_G, rating_PG, rating_PG_13, rating_R, rating_NC_17, rating_UNRATED, rating_OTHER)
- `genre_*`: One-hot encoding of genres from primary/secondary (genre_action, genre_drama, genre_comedy, etc.)

All categorical columns are binarized (0/1) for compatibility with linear models and tree-based algorithms.

## Feature Rationale

The goal is to predict **domestic opening weekend gross** using only information that is knowable *before release* (metadata, planned release timing, and historical performance of related entities like studios and franchises). Features are chosen to capture drivers of demand while minimizing leakage.

### Identity & Time

- `title`
  - Kept for traceability and debugging (not intended as a numeric predictor unless you later add text/vector features).
- `release_year`
  - Captures macro trends: ticket prices, streaming competition, franchise eras, and post-COVID regime shifts that affect openings across years.

### Release Timing (Seasonality)

- `is_summer` (May–Aug)
  - Summer is the blockbuster window; audiences and marketing spend tend to peak here.
- `is_holiday` (Nov 15–Dec 31)
  - Holiday corridor boosts family attendance and repeat viewing; can lift openings for broad-appeal films.
- `is_q1_dump_window` (Jan 1–Feb 20)
  - Often used for lower-confidence releases; historically weaker demand signals.
- `is_spring_dead_zone` (Feb 21–Apr 30)
  - Shoulder season; openings vary and are sensitive to competition and genre.
- `is_awards_season` (Oct 1–Nov 14)
  - Prestige releases can have different rollout strategies and weaker initial openings (platform releases) despite strong long-tail potential.

### Economic Scale

- `log_budget`
  - Budget is a proxy for scale (marketing, wide release likelihood, production values). It's log-transformed because revenue typically grows non-linearly with spend and budget distributions are highly skewed.
- `has_budget`
  - Missing budgets are not random (often smaller/limited releases). This flag lets the model learn a different baseline when budget was imputed vs truly known.
- `runtime_minutes`
  - Proxy for format and audience commitment; also correlates with genre/type (family animation vs epic action) and can influence showtimes/day (capacity).

> Note: budgets are imputed (studio/genre hierarchy) *before* `log_budget` so the transformation stays defined and comparable across rows.

### Content Labels (Discrete Signals)

- `rating_*` (MPA rating one-hot)
  - Rating constrains audience size (e.g., R can reduce family turnout; PG-13 often maximizes blockbuster reach). One-hot encoding avoids imposing an arbitrary ordering.
- `genre_*` (genre one-hot for primary/secondary)
  - Genre strongly impacts opening patterns (horror tends to be front-loaded, family can be steadier, drama often opens smaller). One-hot encoding allows non-linear interactions with seasonality and studio type.

> Implementation note: genre columns can have up to **two 1s** per movie (primary and/or secondary genre).

### Studio & Market Access

- `is_major_studio`
  - Major studios typically secure wider distribution and larger marketing campaigns, increasing opening weekends.
- `is_english`
  - English-language films usually have broader domestic market penetration; non-English films often release narrower initially.

### Star Power (Director & Lead Actor History)

- `director_has_prior` / `lead_actor_has_prior`
  - Binary flags indicating whether director/lead has prior feature film credits.
- `director_prior_count` / `lead_actor_prior_count`
  - Number of prior films, capturing experience and track record depth.
- `director_recent_opening` / `lead_actor_recent_opening` (conditional)
  - Most recent opening weekend from prior films. Only computed when `FETCH_RECENT_OPENING_FEATURES=True` due to expensive API/scraping calls.

### Franchise Signal (History Without Leakage)

- `is_collection`
  - Sequels/franchise entries generally have higher awareness and more predictable turnout.
- `collection_past_open_avg`
  - Captures the franchise's established opening power by averaging **prior** films only (chronological expanding mean with shift), avoiding information from the current film's own opening.

## Complete Feature Table

| Feature | Type | Description | Values |
|---------|------|-------------|--------|
| `title` | String | Movie title | N/A |
| `release_year` | Integer | Year of theatrical release | 2007-2025 |
| `is_summer` | Binary | Released May 1–Aug 31 | 0/1 |
| `is_holiday` | Binary | Released Nov 15–Dec 31 | 0/1 |
| `is_q1_dump_window` | Binary | Released Jan 1–Feb 20 | 0/1 |
| `is_spring_dead_zone` | Binary | Released Feb 21–Apr 30 | 0/1 |
| `is_awards_season` | Binary | Released Oct 1–Nov 14 | 0/1 |
| `log_budget` | Float | log1p(budget_usd) after imputation | Continuous |
| `has_budget` | Binary | 1 if TMDB provided budget; 0 if imputed | 0/1 |
| `runtime_minutes` | Float | Film length in minutes | Continuous (≥60) |
| `rating_G` | Binary | MPA rating: G | 0/1 |
| `rating_PG` | Binary | MPA rating: PG | 0/1 |
| `rating_PG_13` | Binary | MPA rating: PG-13 | 0/1 |
| `rating_R` | Binary | MPA rating: R | 0/1 |
| `rating_NC_17` | Binary | MPA rating: NC-17 | 0/1 |
| `rating_UNRATED` | Binary | No rating or not rated | 0/1 |
| `rating_OTHER` | Binary | International/unrecognized rating | 0/1 |
| `genre_*` | Binary | One-hot for ~18 genres (e.g., action, drama, comedy) | 0/1 each |
| `is_major_studio` | Binary | Distributed by Big 5 + Lionsgate + A24 | 0/1 |
| `is_english` | Binary | Original language is English | 0/1 |
| `is_collection` | Binary | Part of a franchise/collection | 0/1 |
| `collection_past_open_avg` | Float | Mean opening weekend of prior franchise films | Continuous ($USD) |
| `director_has_prior` | Binary | Director has prior feature film credits | 0/1 |
| `director_prior_count` | Float | Number of prior films directed | Continuous |
| `lead_actor_has_prior` | Binary | Lead actor has prior feature film credits | 0/1 |
| `lead_actor_prior_count` | Float | Number of prior films for lead actor | Continuous |
| `director_recent_opening`* | Float | Most recent opening weekend from director's prior films | Continuous ($USD) |
| `lead_actor_recent_opening`* | Float | Most recent opening weekend from lead actor's prior films | Continuous ($USD) |
| `opening_weekend` | Float | **Target**: Domestic opening weekend gross (USD) | Continuous ($USD) |

**Total Features**: ~35-40 (exact count depends on unique genres in dataset and whether recent-opening features are enabled)

**Notes**:
- Rating columns are mutually exclusive (exactly one rating per movie).
- Genre columns allow up to 2 active per movie (primary and/or secondary genre matches).
- `log_budget` uses studio+genre hierarchy imputation before transformation.
- `collection_past_open_avg` only uses prior films (no data leakage).
- All binary features are 0/1 for model compatibility.
- *`director_recent_opening` and `lead_actor_recent_opening` are only included when `FETCH_RECENT_OPENING_FEATURES=True`.

## Key Design Decisions

### Dual-Source Scraping with Fallback

Box Office Mojo is the industry-standard source but has gaps (particularly for limited releases and older films). The Numbers provides broader coverage but lower data quality. The pipeline tries BOM first via IMDb ID, then falls back to The Numbers via title+year slug matching.

### HTML Caching

Web scraping is rate-limited to avoid overloading servers. All HTML pages are cached locally in `cache_html/` using SHA-1 hashes of URLs. On subsequent runs, cached pages are reused, eliminating redundant HTTP requests.

### Star Schema Over Flat Files

Flat CSVs are convenient for quick analysis but become unmaintainable at scale. The star schema provides:

- Referential integrity through foreign keys
- Space efficiency (dimension values stored once, not repeated)
- Query flexibility (complex aggregations without massive JOINs)
- Clear data lineage (staging → warehouse → analytics)

### Separate Training and Prediction Datasets

Avatar 3 is excluded from `preprocessed_movies_all.csv` to prevent data leakage. Its features are computed identically and stored in `preprocessed_avatar3_only.csv` with `opening_weekend = NaN`.

## Data Validation

Every stage includes automated validation:

- No missing values in required columns (budget, runtime, release date)
- Binary flags are strictly 0 or 1 (no unexpected values)
- Opening weekend revenue is positive and non-null for training data
- Dates fall within expected ranges
- Foreign keys reference valid dimension records

## Quick Start

### Prerequisites
```bash
pip install pandas numpy requests beautifulsoup4 lxml python-dateutil tqdm
export TMDB_API_KEY="your_api_key_here"  # Get free key at themoviedb.org
```
### Configuration

Edit these constants at the top of the notebook to control dataset size and data quality:
```python
START_YEAR: int = 2007                          # inclusive
END_YEAR: int = 2025                            # inclusive
MIN_VOTE_COUNT: int = 80                        # TMDB filter
PER_YEAR_LIMIT: int = 1000                      # max movies per year
MIN_RUNTIME_MINUTES: int = 60                   # exclude short films
MIN_OPENING_WEEKEND_USD: int = 1000             # exclude BOM placeholder values
```

**Data Quality Thresholds**:
- Movies with runtime < 60 minutes are excluded (short films, TV specials)
- Movies with opening weekend < $1,000 are excluded (Box Office Mojo placeholder values like $1, $21, $100)

Runtime will scale linearly with the number of movies.

### Run the Pipeline

Execute all cells sequentially. The pipeline will:

1. Fetch movie metadata from TMDB (requires API key)
2. Scrape opening weekend revenue from BOM and The Numbers
3. Build star schema CSVs in `star_schema_output/`
4. Generate flattened raw datasets
5. Engineer features and produce ML-ready CSVs

All outputs are saved to `star_schema_output/` directory.

## Output Files

| File | Description |
|------|-------------|
| `stg_tmdb.csv` | Raw TMDB metadata |
| `stg_opening.csv` | Scraped opening weekend data |
| `dim_movie.csv` | Movie dimension |
| `dim_date.csv` | Date dimension |
| `dim_rating.csv` | Rating dimension |
| `dim_studio.csv` | Studio dimension |
| `bridge_movie_studio.csv` | Movie-studio relationships |
| `fact_opening_weekend.csv` | Opening revenue facts |
| `raw_movies_all.csv` | Denormalized dataset |
| `preprocessed_movies_all.csv` | Training data (ML-ready) |
| `preprocessed_avatar3_only.csv` | Prediction target (ML-ready) |

## Notebook Sections

**Section 0: Imports & Global Configuration** - API keys, year ranges, studio tokens, season windows

**Section 1: TMDB Helpers & Utilities** - API wrapper, movie discovery, date parsing

**Section 2: HTML Caching & Box Office Parsing** - HTML caching, BOM/Numbers parsers

**Section 3: Rating, Studio & Season Helpers** - MPA rating extraction, studio canonicalization, season detection

**Section 4: TMDB Staging** - Discover and fetch movie metadata → `stg_tmdb.csv`

**Section 5: Opening Weekend Staging** - Scrape BOM and Numbers with fallback → `stg_opening.csv`

**Section 6: Dimensions** - Build `dim_date`, `dim_rating`, `dim_studio`, `dim_movie`, `bridge_movie_studio`

**Section 7: FACT – Opening Weekend** - Join staging to dimensions → `fact_opening_weekend.csv`

**Section 8: FLAT RAW DATASET** - Denormalize all tables → `raw_movies_all.csv`

**Section 9: RAW AVATAR 3 ONLY** - Fetch Avatar 3 metadata → `raw_avatar3_only.csv`

**Section 10: Feature Engineering Helpers** - Collection averages, validation functions

**Section 11: Feature Engineering Pipeline** - Engineer 30+ features with validation → `preprocessed_movies_all.csv`, `preprocessed_avatar3_only.csv`

**Section 12: Main** - Orchestrates entire pipeline, optional cache cleanup

**Runtime estimate**: Will vary based on configuration.

## Section 0 - Imports and Global Config

This section defines all imports, API keys, file paths, and configuration constants used throughout the pipeline.

**Imports**:
- `os` - file system operations and environment variables
- `re` - regular expressions for text parsing
- `time` - sleep delays for rate limiting
- `hashlib` - SHA-1 hashing for HTML cache filenames
- `unicodedata` - normalize text encoding
- `ast` - safely parse string representations of Python literals (for CSV deserialization)
- `datetime` - date manipulation and calendar operations
- `typing` - type hints for function signatures
- `numpy` - numerical operations and NaN handling
- `pandas` - dataframe operations and CSV I/O
- `requests` - HTTP client for API calls and web scraping
- `BeautifulSoup` - HTML parsing for box office data
- `dateutil.parser` - flexible date string parsing
- `tqdm` - progress bars for long-running operations
- `concurrent.futures` - parallel execution with ThreadPoolExecutor

**Configuration**:
- Year range and discovery filters: START_YEAR, END_YEAR, MIN_VOTE_COUNT, PER_YEAR_LIMIT
- Runtime and opening weekend filters: MIN_RUNTIME_MINUTES (exclude shorts), MIN_OPENING_WEEKEND_USD (exclude placeholder values from BOM)
- Parallelism and retries: MAX_WORKERS (API + scraping) and MAX_RETRIES; rate-limited sleeps live inside staging functions
- Incremental controls: INCREMENTAL_TMDB / INCREMENTAL_OPENING reuse existing staging CSVs; FORCE_REFRESH_TMDB / FORCE_REFRESH_OPENING force full rebuilds
- Target ID: AVATAR3_TMDB_ID for downstream prediction examples
- Paths and cache: DATA_DIR as output root, CACHE_DIR for HTML cache, DELETE_CACHE_ON_FINISH toggle
- Networking: TMDB_BASE and shared SESSION with custom User-Agent
- Classification helpers: MAJOR_STUDIOS_TOKENS for studio tagging; SEASON_WINDOWS for release-window features

All configuration values are parameterized for easy modification.

In [15]:
import os
import re
import time
import hashlib
import unicodedata
import ast
from datetime import date, timedelta
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as dtparser
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# --------------------
# 0.1 – API Keys & Ranges
# --------------------

TMDB_KEY: str = os.getenv("TMDB_API_KEY", "5eb38b42bd7adf22c83cce6592f02f8c")

START_YEAR: int = 2007       # inclusive
END_YEAR: int = 2025        # inclusive
MIN_VOTE_COUNT: int = 80     # TMDB filter
PER_YEAR_LIMIT: int = 1000     # max movies per year
MIN_RUNTIME_MINUTES: int = 60  # minimum runtime to exclude short films
MIN_OPENING_WEEKEND_USD: int = 1000  # minimum opening weekend to exclude placeholders
MAX_WORKERS: int = 10         # parallel workers for API calls and scraping
MAX_RETRIES: int = 3         # retry attempts for failed requests

# --------------------
# 0.1a – OPTIONAL EXPENSIVE FEATURES
# --------------------
# Set to True to compute director_recent_opening and lead_actor_recent_opening features
# These require additional API calls and web scraping, so set to False for faster runs
FETCH_RECENT_OPENING_FEATURES: bool = True

# --------------------
# 0.1b – Incremental controls
# --------------------
INCREMENTAL_TMDB: bool = True        # reuse existing stg_tmdb.csv and only fetch missing IDs
INCREMENTAL_OPENING: bool = True     # reuse existing stg_opening.csv and only scrape missing IDs
FORCE_REFRESH_TMDB: bool = False     # ignore existing TMDB staging when True
FORCE_REFRESH_OPENING: bool = False  # ignore existing opening staging when True
REFRESH_CACHE: bool = False          # force re-fetch of cached person credits and openings

AVATAR3_TMDB_ID: int = 83533

# --------------------
# 0.2 – Paths & HTTP Session
# --------------------

DATA_DIR: str = "star_schema_output"
CACHE_DIR: str = os.path.join(DATA_DIR, "cache_html")
PERSON_CACHE_DIR: str = os.path.join(DATA_DIR, "cache_person")
OPENING_CACHE_DIR: str = os.path.join(DATA_DIR, "cache_opening")
DELETE_CACHE_ON_FINISH: bool = False

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PERSON_CACHE_DIR, exist_ok=True)
os.makedirs(OPENING_CACHE_DIR, exist_ok=True)

TMDB_BASE: str = "https://api.themoviedb.org/3"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 (compatible; StarSchemaBoxOffice/1.0)"})


# --------------------
# 0.3 – Major studio tokens
# --------------------
MAJOR_STUDIOS_TOKENS = [
    # DISNEY
    "walt disney",
    "waltdisney",
    "disney",
    "pixar",
    "marvel",
    "lucasfilm",
    "buena vista",
    "20th century studios",
    "searchlight",
    "touchstone",
    "walt disney animation",
    "walt disney studios motion",

    # WARNER
    "warner bros",
    "warnerbros",
    "warner brothers",
    "new line cinema",
    "newline cinema",
    "turner entertainment",

    # UNIVERSAL
    "universal pictures",
    "universal studios",
    "focus features",
    "dreamworks",
    "illumination",

    # SONY
    "sony pictures",
    "columbia pictures",
    "tristar",
    "tri-star",
    "screen gems",
    "sony pictures releasing",

    # PARAMOUNT
    "paramount",
    "paramount pictures",
    "nickelodeon movies",
    "miramax",
    "skydance",

    # LIONSGATE
    "lionsgate",
    "summit entertainment",
    "starz",
    "grindstone entertainment",

    # AMAZON / MGM
    "amazon studios",
    "amazon mgm studios",
    "metro-goldwyn-mayer",
    "mgm",
    "orion pictures",
    "united artists",

    # A24
    "a24",
]


# --------------------
# 0.4 – Season windows
# --------------------
SEASON_WINDOWS = {
    "q1_dump_window": {  # Jan 1 – Feb 20
        "start": (1, 1),
        "end": (2, 20),
    },
    "spring_dead_zone": {  # Feb 21 – Apr 30
        "start": (2, 21),
        "end": (4, 30),
    },
    "summer": {  # May 1 – Aug 31
        "start": (5, 1),
        "end": (8, 31),
    },
    "awards_season": {  # Oct 1 – Nov 14
        "start": (10, 1),
        "end": (11, 14),
    },
    "holiday": {  # Nov 15 – Dec 31
        "start": (11, 15),
        "end": (12, 31),
    },

}


## Section 1 - TMDB Helpers and Utilities

This section provides wrapper functions for interacting with the TMDB API and utility functions for data normalization.

**API Functions**:
- `retry_with_backoff()` - Decorator that retries API calls on transient failures with exponential backoff (1s, 2s, 4s), logging each attempt and re-raising after MAX_RETRIES
- `_require_tmdb_key()` - Validates TMDB API key is configured, raises error if missing or invalid
- `tmdb_get()` - Generic TMDB API wrapper with automatic retry logic, detects authentication method by key length (Bearer token if >40 chars, otherwise API key parameter), 30-second timeout
- `discover_movies_by_year()` - Discovers movies for a given year sorted by revenue, filters by MIN_VOTE_COUNT, excludes adult content, handles pagination with 0.1s rate limiting, displays progress bar for page iteration, returns up to PER_YEAR_LIMIT movies
- `tmdb_movie_full()` - Fetches complete movie details with appended credits, external IDs (IMDb), and release dates in a single API call

**Utility Functions**:
- `safe_year()` - Extracts year from date string, returns NaN if parsing fails (prevents pipeline crashes on malformed dates)
- `clean_title()` - Normalizes Unicode (NFKD) and collapses whitespace in movie titles

In [16]:
def retry_with_backoff(func):
    """Decorator that retries a function with exponential backoff on failure."""
    def wrapper(*args, **kwargs):
        for attempt in range(MAX_RETRIES):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    raise
                wait_time = 2 ** attempt  # 1s, 2s, 4s
                print(f"[RETRY] Attempt {attempt + 1}/{MAX_RETRIES} failed: {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
        return None
    return wrapper


def _require_tmdb_key() -> None:
    if not TMDB_KEY or TMDB_KEY.startswith("YOUR_") or len(TMDB_KEY.strip()) < 10:
        raise RuntimeError(
            "TMDB API key is not configured.\n"
            "Set the TMDB_API_KEY environment variable or edit TMDB_KEY in this script."
        )


@retry_with_backoff
def tmdb_get(path: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    _require_tmdb_key()
    url = f"{TMDB_BASE}{path}"
    params = dict(params or {})

    if len(TMDB_KEY) > 40:
        headers = {"Authorization": f"Bearer {TMDB_KEY}"}
        resp = SESSION.get(url, params=params, headers=headers, timeout=30)
    else:
        params["api_key"] = TMDB_KEY
        resp = SESSION.get(url, params=params, timeout=30)

    resp.raise_for_status()
    return resp.json()


def discover_movies_by_year(year: int) -> List[Dict[str, Any]]:
    movies: List[Dict[str, Any]] = []

    for page in tqdm(range(1, 50), desc=f"Discovering {year}", leave=False):
        data = tmdb_get("/discover/movie", {
            "sort_by": "revenue.desc",
            "primary_release_date.gte": f"{year}-01-01",
            "primary_release_date.lte": f"{year}-12-31",
            "vote_count.gte": MIN_VOTE_COUNT,
            "page": page,
            "include_adult": "false",
        })
        movies.extend(data.get("results", []))

        if page >= data.get("total_pages", 1):
            break
        if len(movies) >= PER_YEAR_LIMIT:
            break

        time.sleep(0.1)

    return movies[:PER_YEAR_LIMIT]


def tmdb_movie_full(movie_id: int) -> Dict[str, Any]:
    return tmdb_get(
        f"/movie/{movie_id}",
        {"append_to_response": "credits,external_ids,release_dates"}
    )


def safe_year(date_str: Optional[str]) -> float:
    if not date_str:
        return np.nan
    try:
        return float(dtparser.parse(date_str).year)
    except Exception:
        return np.nan


def clean_title(title: Optional[str]) -> str:
    s = unicodedata.normalize("NFKD", (title or "").strip())
    return re.sub(r"\s+", " ", s)


def title_slug_for_numbers(title: str, year: int) -> str:
    t = clean_title(title).replace("&", "and")
    t = re.sub(r"[^\w\s:()-]", "", t)
    t = re.sub(r"\s+", "-", t.strip())
    return f"{t}-({int(year)})"

## Section 2 - HTML Caching & Box Office Parsing

This section handles web scraping with local caching and parses opening weekend revenue from Box Office Mojo and The Numbers.

**Caching Functions**:
- `cache_path()` - Generates cache filename from URL using SHA-1 hash, stores in CACHE_DIR as .html files
- `fetch_html()` - Fetches HTML from URL with caching (checks cache first, only hits network if not cached), attempts up to MAX_RETRIES requests with 30-second timeout and exponential backoff (1s, 2s, 4s), returns empty string if all attempts fail

**Parsing Functions**:
- `parse_bom_opening()` - Extracts opening weekend revenue from Box Office Mojo HTML, tries multiple key patterns in order ("Domestic Opening", "Opening Weekend", "Opening (Domestic)", then generic "Opening"), uses case-insensitive regex to find dollar amounts, removes commas and converts to float, returns NaN if not found
- `parse_numbers_opening()` - Extracts opening weekend revenue from The Numbers HTML, searches for "Opening Weekend" followed by dollar amount with case-insensitive regex, removes commas and converts to float, returns NaN if not found

**URL Construction**:
- `bom_url_from_imdb()` - Builds Box Office Mojo URL from IMDb ID (format: boxofficemojo.com/title/{imdb_id}/)
- `numbers_url_from_title_year()` - Builds The Numbers URL from title and year using slug format (format: the-numbers.com/movie/{slug}#tab=box-office)

All HTML parsing uses BeautifulSoup with lxml parser and joins all stripped text into a single string for pattern matching.

In [17]:
def cache_path(url: str) -> str:
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()
    return os.path.join(CACHE_DIR, f"{h}.html")


def fetch_html(url: str, retries: int = None, sleep: float = 1.0) -> str:
    path = cache_path(url)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

    if retries is None:
        retries = MAX_RETRIES

    for attempt in range(retries):
        try:
            resp = SESSION.get(url, timeout=30)
            if resp.status_code == 200 and resp.text:
                with open(path, "w", encoding="utf-8") as f:
                    f.write(resp.text)
                return resp.text
        except Exception as e:
            if attempt < retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                print(f"[RETRY] HTML fetch failed for {url[:50]}... Attempt {attempt + 1}/{retries}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                time.sleep(sleep)

    return ""


def parse_bom_opening(html: str) -> float:
    if not html:
        return np.nan
    text = " ".join(BeautifulSoup(html, "lxml").stripped_strings)

    for key in ("Domestic Opening", "Opening Weekend", "Opening (Domestic)"):
        m = re.search(fr"{key}[^$]*\$\s*([0-9][0-9,]*)", text, flags=re.I)
        if m:
            return float(m.group(1).replace(",", ""))

    m2 = re.search(r"Opening[^$]*\$\s*([0-9][0-9,]*)", text, flags=re.I)
    if m2:
        return float(m2.group(1).replace(",", ""))

    return np.nan


def parse_numbers_opening(html: str) -> float:
    if not html:
        return np.nan
    text = " ".join(BeautifulSoup(html, "lxml").stripped_strings)
    m = re.search(r"Opening Weekend[^$]*\$\s*([0-9][0-9,]*)", text, flags=re.I)
    if m:
        return float(m.group(1).replace(",", ""))
    return np.nan


def bom_url_from_imdb(imdb_id: str) -> str:
    return f"https://www.boxofficemojo.com/title/{imdb_id.strip()}/"


def numbers_url_from_title_year(title: str, year: int) -> str:
    slug = title_slug_for_numbers(title, year)
    return f"https://www.the-numbers.com/movie/{slug}#tab=box-office"

## Section 3 - Rating, Studio & Season Helpers

This section provides utility functions for extracting MPA ratings, canonicalizing studio names, classifying major studios, and detecting seasonal release windows.

**Rating Functions**:
- `extract_mpa_rating_from_movie_dict()` - Extracts MPA rating from TMDB movie dictionary's release_dates field, prioritizes US certification, falls back to first available certification from any country if US not found, returns None if no certifications exist
- `normalize_rating()` - Normalizes 100+ international rating variants to 7 standard US MPA categories (G, PG, PG-13, R, NC-17, UNRATED, OTHER), handles Korean, European, Latin American ratings, TV ratings, and deprecated US ratings

**Studio Functions**:
- `studio_name_canonical()` - Canonicalizes studio names by converting to lowercase and normalizing whitespace (e.g., "Walt  Disney" → "walt disney")
- `is_major_studio_from_text()` - Checks if text contains any major studio token from MAJOR_STUDIOS_TOKENS list, returns 1 if match found (case-insensitive substring search), returns 0 if no match or invalid input

**Season Functions**:
- `in_month_day_range()` - Checks if a (month, day) tuple falls within a given start and end range (unused helper function)
- `add_season_flags()` - Adds five binary season columns to dataframe based on date column (is_summer, is_holiday, is_q1_dump_window, is_spring_dead_zone, is_awards_season), uses SEASON_WINDOWS configuration, handles null dates gracefully by setting flags to 0, returns modified dataframe

Season flags are non-overlapping binary indicators (0 or 1) based on month and day only (year-agnostic). Some dates may not fall into any season window (e.g., September).

In [18]:
def extract_mpa_rating_from_movie_dict(movie: Dict[str, Any]) -> Optional[str]:
    rel = movie.get("release_dates") or {}
    results = rel.get("results") or []

    first_fallback: Optional[str] = None

    for country_data in results:
        iso = country_data.get("iso_3166_1")
        dates = country_data.get("release_dates") or []
        for rd in dates:
            cert = (rd.get("certification") or "").strip()
            if not cert:
                continue
            if iso == "US":
                return cert
            if first_fallback is None:
                first_fallback = cert
    return first_fallback

def studio_name_canonical(name: str) -> str:
    return " ".join(name.strip().lower().split())


def is_major_studio_from_text(text: Any) -> int:
    if not isinstance(text, str) or not text.strip():
        return 0
    canon = text.lower()
    return int(any(token in canon for token in MAJOR_STUDIOS_TOKENS))


def in_month_day_range(month: int, day: int, start: Tuple[int, int], end: Tuple[int, int]) -> bool:
    sm, sd = start
    em, ed = end
    return (month, day) >= (sm, sd) and (month, day) <= (em, ed)


def add_season_flags(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    dt = pd.to_datetime(df[date_col], errors="coerce")
    month = dt.dt.month
    day = dt.dt.day

    def flag(name: str) -> pd.Series:
        cfg = SEASON_WINDOWS[name]
        sm, sd = cfg["start"]
        em, ed = cfg["end"]
        return ((month.notna()) & (day.notna()) &
                ((month > sm) | ((month == sm) & (day >= sd))) &
                ((month < em) | ((month == em) & (day <= ed)))).astype(int)

    df["is_summer"] = flag("summer")
    df["is_holiday"] = flag("holiday")
    df["is_q1_dump_window"] = flag("q1_dump_window")
    df["is_spring_dead_zone"] = flag("spring_dead_zone")
    df["is_awards_season"] = flag("awards_season")

    return df


def normalize_rating(rating_code: Any) -> str:
    """
    Normalize rating codes to standard US MPA categories.
    Maps 100+ international rating variants to 7 standard categories:
    G, PG, PG-13, R, NC-17, UNRATED, OTHER
    """
    if pd.isna(rating_code) or not isinstance(rating_code, str):
        return "UNRATED"
    
    code = str(rating_code).strip().upper()
    
    if not code or code in ["-", "UNKNOWN"]:
        return "UNRATED"
    
    # G ratings (General Audiences - All ages)
    if code in ["G", "GP", "TV-G", "U", "UC", "E", "ALL", "AL", "L", "APTA", 
                "ATP", "GENEL İZLEYİCİ", "A", "AA", "AP", "0", "EA", "KN",
                "Κ", "К", "전체관람가"]:
        return "G"
    
    # PG ratings (Parental Guidance - some material may not be suitable for children)
    if code in ["PG", "TV-PG", "M", "M/PG", "6", "6+", "7", "7+", "8", "9", "10",
                "I", "IIA", "B", "C", "T", "TP", "S", "Κ-12", "N-7", "K-12", 
                "K12", "E 12"]:
        return "PG"
    
    # PG-13 ratings (Parents Strongly Cautioned - some material inappropriate for under 13)
    if code in ["PG-13", "PG13", "TV-14", "12", "12+", "12A", "PG12", "11", "13", "13+",
                "14", "14+", "14A", "15", "15+", "15A", "M/12", "M/14", "IIB", "III",
                "12 ANOS", "14 ANOS", "E 14", "VM14", "B-15", "N-13", "R-13",
                "12세 이상 관람가", "15세 이상 관람가", "15세이상관람가",
                "K-15", "K15", "Κ-15", "R15+", "MA 15+", "MA15+"]:
        return "PG-13"
    
    # R ratings (Restricted - under 17 requires parent/guardian)
    if code in ["R", "TV-MA", "16", "16+", "17+", "18A", "16 ANOS",
                "K-16", "K16", "N-16", "NC16", "Κ-18"]:
        return "R"
    
    # NC-17 ratings (No one 17 and under admitted)
    if code in ["NC-17", "NC17", "X", "18", "18+", "19", "R 18+", "R18+"]:
        return "NC-17"
    
    # Unrated / Not Rated
    if code in ["NR", "NOT RATED", "UNRATED", "APPROVED", "PASSED", "UA",
                "BTL", "KT", "KT/EA", "ENA", "ATP", "ALL"]:
        return "UNRATED"
    
    # Everything else goes to OTHER (rare/unknown international ratings)
    return "OTHER"

## Section 4 - TMDB Stage

This section discovers movies from TMDB across the configured year range and fetches complete metadata for each movie, producing the raw staging table.

**Incremental Mode**:
- When `INCREMENTAL_TMDB=True` (and not `FORCE_REFRESH_TMDB`), reuse `stg_tmdb.csv`, skip tmdb_ids already present, and only fetch the missing movies; otherwise fetch all.

**Process**:
1. Discovers movies for each year from START_YEAR to END_YEAR using `discover_movies_by_year()` with progress bar showing year-by-year completion
2. Collects unique movie IDs (deduplicates across years), then filters out already-seen IDs when incremental mode is on
3. Fetches full details for remaining movies using `tmdb_movie_full()` in parallel with MAX_WORKERS (5) threads via ThreadPoolExecutor, with progress bar displaying iteration speed, ETA, and completion percentage
4. Extracts and transforms metadata:
   - Basic info: tmdb_id, imdb_id, title, original_title, release_date, language, runtime, status
   - Financial: budget_usd
   - Metrics: popularity, vote_average, vote_count
   - Genres: primary_genre (first), secondary_genre (second)
   - Collection: collection_name (franchise membership)
   - Rating: mpa_rating_raw (extracted using `extract_mpa_rating_from_movie_dict()`)
   - Studios: production_companies_raw (list of dicts), production_companies (comma-separated string)
   - Popularity metrics: director_popularity (mean of all directors), cast_popularity (mean of top 5 cast)
5. Sleeps 0.1 seconds per API call for rate limiting (parallelized with 5 workers for ~5x speedup)
6. Merges with any prior staging data (if reused), drops duplicate tmdb_ids keeping the latest
7. Coerces numeric columns to float type
8. Saves to `stg_tmdb.csv`

**Error Handling**: If fetching details for a movie fails, logs error and continues with remaining movies.

**Parallel Processing**: Uses ThreadPoolExecutor with MAX_WORKERS=5 for concurrent API requests, significantly reducing total execution time while respecting rate limits.


In [19]:
def build_tmdb_staging() -> pd.DataFrame:
    tmdb_path = os.path.join(DATA_DIR, "stg_tmdb.csv")

    existing_df = pd.DataFrame()
    existing_ids: set[int] = set()

    if INCREMENTAL_TMDB and not FORCE_REFRESH_TMDB and os.path.exists(tmdb_path):
        try:
            existing_df = pd.read_csv(tmdb_path)
            if "tmdb_id" in existing_df.columns:
                existing_ids = set(
                    pd.to_numeric(existing_df["tmdb_id"], errors="coerce")
                    .dropna()
                    .astype(int)
                    .tolist()
                )
            print(f"[TMDB] Loaded existing staging rows: {len(existing_df):,}")
        except Exception as e:
            print(f"[TMDB] Warning: could not load existing staging ({e}); rebuilding from scratch.")
            existing_df = pd.DataFrame()
            existing_ids = set()

    print(f"Discovering TMDB titles from {START_YEAR} to {END_YEAR} ...")
    discovered_ids: List[int] = []

    for year in tqdm(range(START_YEAR, END_YEAR + 1), desc="Years", unit="year"):
        movies = discover_movies_by_year(year)
        discovered_ids.extend([m["id"] for m in movies])

    unique_ids = sorted(set(discovered_ids))

    if INCREMENTAL_TMDB and not FORCE_REFRESH_TMDB:
        remaining_ids = [mid for mid in unique_ids if mid not in existing_ids]
    else:
        remaining_ids = unique_ids

    print(f"[TMDB] Unique discovered movies: {len(unique_ids)}")
    print(f"[TMDB] Remaining to fetch this run: {len(remaining_ids)}")

    rows: List[Dict[str, Any]] = []

    def fetch_movie_details(mid: int) -> Optional[Dict[str, Any]]:
        try:
            d = tmdb_movie_full(mid)

            genres = d.get("genres") or []
            genre_names = [g.get("name") for g in genres if g.get("name")]
            primary_genre = genre_names[0] if genre_names else None
            secondary_genre = genre_names[1] if len(genre_names) > 1 else None

            belongs = d.get("belongs_to_collection")
            collection_name = belongs.get("name") if isinstance(belongs, dict) else None

            mpa_raw = extract_mpa_rating_from_movie_dict(d)

            prod_raw = d.get("production_companies") or []
            prod_names = [pc.get("name") for pc in prod_raw if pc.get("name")]
            prod_names_str = ", ".join(prod_names) if prod_names else None

            credits = d.get("credits") or {}
            crew = credits.get("crew") or []
            cast = credits.get("cast") or []

            directors = [c for c in crew if c.get("job") == "Director"]
            director_popularity = np.nanmean(
                [x.get("popularity", np.nan) for x in directors]
            ) if directors else np.nan

            top_cast = cast[:5]
            cast_popularity = np.nanmean(
                [x.get("popularity", np.nan) for x in top_cast]
            ) if top_cast else np.nan
            max_cast_popularity = np.nanmax(
                [x.get("popularity", np.nan) for x in top_cast]
            ) if top_cast else np.nan

            time.sleep(0.1)

            return {
                "tmdb_id": d.get("id"),
                "imdb_id": (d.get("external_ids") or {}).get("imdb_id"),
                "title": d.get("title"),
                "original_title": d.get("original_title"),
                "release_date_raw": d.get("release_date"),
                "original_language": d.get("original_language"),
                "runtime_minutes": d.get("runtime"),
                "status": d.get("status"),
                "budget_usd": d.get("budget"),
                "popularity": d.get("popularity"),
                "vote_average": d.get("vote_average"),
                "vote_count": d.get("vote_count"),
                "primary_genre": primary_genre,
                "secondary_genre": secondary_genre,
                "collection_name": collection_name,
                "mpa_rating_raw": mpa_raw,
                "production_companies_raw": prod_raw,
                "production_companies": prod_names_str,
                "director_popularity": director_popularity,
                "cast_popularity": cast_popularity,
                "max_cast_popularity": max_cast_popularity,
            }
        except Exception as e:
            print(f"[TMDB] Error fetching details for {mid}: {e}")
            return None

    if remaining_ids:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(fetch_movie_details, int(mid)): mid for mid in remaining_ids}

            for future in tqdm(
                as_completed(futures),
                total=len(remaining_ids),
                desc="Fetching TMDB movie details",
                unit="movie",
            ):
                result = future.result()
                if result is not None:
                    rows.append(result)
    else:
        print("[TMDB] Nothing new to fetch; reusing existing staging.")

    new_df = pd.DataFrame(rows)
    if not new_df.empty:
        new_df = new_df.drop_duplicates(subset=["tmdb_id"])

    combined_df = existing_df.copy() if not existing_df.empty else pd.DataFrame()
    if not new_df.empty:
        combined_df = pd.concat([combined_df, new_df], ignore_index=True) if not combined_df.empty else new_df

    expected_cols = [
        "tmdb_id",
        "imdb_id",
        "title",
        "original_title",
        "release_date_raw",
        "original_language",
        "runtime_minutes",
        "status",
        "budget_usd",
        "popularity",
        "vote_average",
        "vote_count",
        "primary_genre",
        "secondary_genre",
        "collection_name",
        "mpa_rating_raw",
        "production_companies_raw",
        "production_companies",
        "director_popularity",
        "cast_popularity",
        "max_cast_popularity",
    ]
    if combined_df.empty:
        combined_df = pd.DataFrame(columns=expected_cols)

    combined_df = combined_df.drop_duplicates(subset=["tmdb_id"], keep="last")

    numeric_cols = [
        "budget_usd",
        "popularity",
        "vote_average",
        "vote_count",
        "runtime_minutes",
        "director_popularity",
        "cast_popularity",
        "max_cast_popularity",
    ]
    for col in numeric_cols:
        if col in combined_df.columns:
            combined_df[col] = pd.to_numeric(combined_df[col], errors="coerce").astype(float)

    # Filter to feature-length films only (>=60 min)
    # Excludes short films, TV specials, and incomplete entries
    before_filter = len(combined_df)
    combined_df = combined_df[combined_df["runtime_minutes"] >= MIN_RUNTIME_MINUTES]
    excluded_shorts = before_filter - len(combined_df)

    combined_df.to_csv(tmdb_path, index=False)

    # === DATA QUALITY SUMMARY ===
    print("\n" + "="*70)
    print("TMDB STAGING SUMMARY")
    print("="*70)
    print(f"Movies fetched this run: {len(new_df):,}")
    print(f"Total movies in staging: {len(combined_df):,}")
    if excluded_shorts > 0:
        print(f"Excluded {excluded_shorts} short film(s) (<{MIN_RUNTIME_MINUTES} min)")


    # Release date range
    if "release_date_raw" in combined_df.columns and not combined_df.empty:
        valid_dates = pd.to_datetime(combined_df["release_date_raw"], errors="coerce")
        valid_dates = valid_dates[valid_dates.notna()]
        if not valid_dates.empty:
            print(f"Date range: {valid_dates.min().date()} to {valid_dates.max().date()}")

    # Missing data analysis
    if not combined_df.empty:
        missing_imdb = combined_df["imdb_id"].isna().sum()
        missing_budget = combined_df["budget_usd"].isna().sum()
        missing_runtime = combined_df["runtime_minutes"].isna().sum()
        print(f"\nMissing data:")
        print(f"  - IMDb IDs: {missing_imdb} ({missing_imdb/len(combined_df)*100:.1f}%)")
        print(f"  - Budget: {missing_budget} ({missing_budget/len(combined_df)*100:.1f}%)")
        print(f"  - Runtime: {missing_runtime} ({missing_runtime/len(combined_df)*100:.1f}%)")

        # Budget statistics (excluding zeros/nulls)
        valid_budgets = combined_df[combined_df["budget_usd"] > 0]["budget_usd"]
        if not valid_budgets.empty:
            print(f"\nBudget statistics (n={len(valid_budgets)}):")
            print(f"  - Min: ${valid_budgets.min():,.0f}")
            print(f"  - Median: ${valid_budgets.median():,.0f}")
            print(f"  - Mean: ${valid_budgets.mean():,.0f}")
            print(f"  - Max: ${valid_budgets.max():,.0f}")

        # Runtime statistics
        valid_runtimes = combined_df[combined_df["runtime_minutes"] > 0]["runtime_minutes"]
        if not valid_runtimes.empty:
            print(f"\nRuntime statistics (n={len(valid_runtimes)}):")
            print(f"  - Min: {valid_runtimes.min():.0f} min")
            print(f"  - Median: {valid_runtimes.median():.0f} min")
            print(f"  - Max: {valid_runtimes.max():.0f} min")

        # Genre distribution
        genre_counts = combined_df["primary_genre"].value_counts().head(5)
        print(f"\nTop 5 genres:")
        for genre, count in genre_counts.items():
            print(f"  - {genre}: {count} ({count/len(combined_df)*100:.1f}%)")

    print("="*70 + "\n")

    return combined_df


## Section 5 - Opening Weekend Staging

This section scrapes domestic opening weekend box office revenue for each movie using a dual-source strategy with fallback.

**Data Quality Filters**:
- Minimum opening weekend threshold: MIN_OPENING_WEEKEND_USD ($1,000) to exclude implausibly low values
- Rationale: Box Office Mojo uses placeholder values ($1, $21, $100) for international/limited releases that never had proper US theatrical releases
- Only movies with opening_weekend >= $1,000 are retained (even smallest 1-2 theater limited release would exceed this)

**Incremental Mode**:
- When `INCREMENTAL_OPENING=True` (and not `FORCE_REFRESH_OPENING`), reuse `stg_opening.csv`, skip tmdb_ids already scraped, and only process the remaining movies; otherwise scrape all.

**Process**:
1. Filters tmdb_stg to movies not yet scraped when incremental mode is on, then iterates through them in parallel with MAX_WORKERS (5) threads, with progress bar showing scraping progress and estimated time remaining
2. For each movie, attempts to scrape opening weekend revenue using a two-tier approach:
   - **Primary source (Box Office Mojo)**: If movie has valid IMDb ID (starts with "tt"), constructs BOM URL, fetches HTML, parses opening weekend value, sleeps 0.3 seconds
   - **Fallback source (The Numbers)**: If BOM fails or no IMDb ID, and year is valid, constructs The Numbers URL from title and year, fetches HTML, parses opening weekend value, sleeps 0.3 seconds
3. Records movie only if opening weekend value is found and greater than zero
4. Stores metadata: tmdb_id, imdb_id, title, release_year, opening_weekend_gross_usd, opening_source ("bom" or "numbers"), source_url
5. Merges with prior opening staging (if reused), deduplicates by tmdb_id, coerces opening_weekend_gross_usd to float
6. Saves to `stg_opening.csv`

**Scraping Strategy**: Sequential fallback with rate limiting (0.3s sleep after each attempt). HTML is cached locally, so repeated runs reuse cached pages.

**Parallel Processing**: Uses ThreadPoolExecutor with MAX_WORKERS=5 for concurrent scraping, significantly reducing total execution time while respecting rate limits (~4-5x speedup).


In [20]:
def build_opening_staging(tmdb_stg: pd.DataFrame) -> pd.DataFrame:
    opening_path = os.path.join(DATA_DIR, "stg_opening.csv")

    existing_df = pd.DataFrame()
    existing_ids: set[int] = set()

    if INCREMENTAL_OPENING and not FORCE_REFRESH_OPENING and os.path.exists(opening_path):
        try:
            existing_df = pd.read_csv(opening_path)
            if "tmdb_id" in existing_df.columns:
                existing_ids = set(
                    pd.to_numeric(existing_df["tmdb_id"], errors="coerce")
                    .dropna()
                    .astype(int)
                    .tolist()
                )
            print(f"[OPENING] Loaded existing opening rows: {len(existing_df):,}")
        except Exception as e:
            print(f"[OPENING] Warning: could not load existing opening staging ({e}); rebuilding from scratch.")
            existing_df = pd.DataFrame()
            existing_ids = set()

    if INCREMENTAL_OPENING and not FORCE_REFRESH_OPENING:
        todo_df = tmdb_stg[~tmdb_stg["tmdb_id"].isin(existing_ids)]
    else:
        todo_df = tmdb_stg

    if todo_df.empty:
        print("[OPENING] Nothing new to scrape; reusing existing staging.")
    else:
        print("Scraping DOMESTIC opening-weekend box office (BOM -> The Numbers)...")

    records: List[Dict[str, Any]] = []

    def scrape_movie_opening(row_data: tuple) -> Optional[Dict[str, Any]]:
        _, row = row_data
        title = row["title"]
        imdb_id = row.get("imdb_id")
        release_date = row.get("release_date_raw")
        year = safe_year(release_date)

        opening_val = np.nan
        source: Optional[str] = None
        source_url: Optional[str] = None

        if isinstance(imdb_id, str) and imdb_id.startswith("tt"):
            url_bom = bom_url_from_imdb(imdb_id)
            html_bom = fetch_html(url_bom)
            val_bom = parse_bom_opening(html_bom)

            if not np.isnan(val_bom):
                opening_val = val_bom
                source = "bom"
                source_url = url_bom

            time.sleep(0.3)

        if np.isnan(opening_val) and pd.notna(year):
            url_nums = numbers_url_from_title_year(str(title), int(year))
            html_nums = fetch_html(url_nums)
            val_nums = parse_numbers_opening(html_nums)

            if not np.isnan(val_nums):
                opening_val = val_nums
                source = "numbers"
                source_url = url_nums

            time.sleep(0.3)

        if not np.isnan(opening_val) and opening_val >= MIN_OPENING_WEEKEND_USD:
            return {
                "tmdb_id": row["tmdb_id"],
                "imdb_id": imdb_id,
                "title": title,
                "release_year": year,
                "opening_weekend_gross_usd": opening_val,
                "opening_source": source,
                "opening_source_url": source_url,
            }
        return None

    if not todo_df.empty:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(scrape_movie_opening, row_data): row_data for row_data in todo_df.iterrows()}

            for future in tqdm(
                as_completed(futures),
                total=len(todo_df),
                desc="Scraping box office",
                unit="movie",
            ):
                result = future.result()
                if result is not None:
                    records.append(result)

    new_df = pd.DataFrame(records)
    if not new_df.empty:
        new_df["opening_weekend_gross_usd"] = pd.to_numeric(
            new_df["opening_weekend_gross_usd"], errors="coerce"
        ).astype(float)
        new_df = new_df.drop_duplicates(subset=["tmdb_id"])

    combined_df = existing_df.copy() if not existing_df.empty else pd.DataFrame()
    if not new_df.empty:
        combined_df = pd.concat([combined_df, new_df], ignore_index=True) if not combined_df.empty else new_df

    expected_cols = [
        "tmdb_id",
        "imdb_id",
        "title",
        "release_year",
        "opening_weekend_gross_usd",
        "opening_source",
        "opening_source_url",
        "release_date_raw",
    ]
    if combined_df.empty:
        combined_df = pd.DataFrame(columns=expected_cols)

    combined_df = combined_df.drop_duplicates(subset=["tmdb_id"], keep="last")

    if "opening_weekend_gross_usd" in combined_df.columns:
        combined_df["opening_weekend_gross_usd"] = pd.to_numeric(
            combined_df["opening_weekend_gross_usd"], errors="coerce"
        ).astype(float)

    combined_df.to_csv(opening_path, index=False)

    # === BOX OFFICE SCRAPING SUMMARY ===
    print("\n" + "="*70)
    print("BOX OFFICE SCRAPING SUMMARY")
    print("="*70)
    print(f"Movies in TMDB staging: {len(tmdb_stg):,}")
    print(f"Scraped this run: {len(new_df):,}")
    print(f"Total with opening data: {len(combined_df):,}")
    success_rate = len(combined_df)/len(tmdb_stg)*100 if len(tmdb_stg) > 0 else 0
    print(f"Success rate: {success_rate:.1f}%")
    print(f"Note: Excludes movies with opening weekend <${MIN_OPENING_WEEKEND_USD:,} (placeholder values)")

    # Source breakdown
    if len(combined_df) > 0:
        bom_count = (combined_df["opening_source"] == "bom").sum()
        numbers_count = (combined_df["opening_source"] == "numbers").sum()
        print(f"\nData sources:")
        print(f"  - Box Office Mojo: {bom_count} ({bom_count/len(combined_df)*100:.1f}%)")
        print(f"  - The Numbers (fallback): {numbers_count} ({numbers_count/len(combined_df)*100:.1f}%)")

        # Opening weekend statistics
        print(f"\nOpening weekend revenue (n={len(combined_df)}):")
        print(f"  - Min: ${combined_df['opening_weekend_gross_usd'].min():,.0f}")
        print(f"  - Median: ${combined_df['opening_weekend_gross_usd'].median():,.0f}")
        print(f"  - Mean: ${combined_df['opening_weekend_gross_usd'].mean():,.0f}")
        print(f"  - Max: ${combined_df['opening_weekend_gross_usd'].max():,.0f}")

    # Movies that will be excluded (no opening data)
    excluded = len(tmdb_stg) - len(combined_df)
    if excluded > 0:
        print(f"\nWARNING: {excluded} movies will be excluded from training (no opening data)")

    print("="*70 + "\n")

    return combined_df


## Section 6 - Dimensions

This section builds the dimension tables for the star schema, including date, rating, studio, and movie dimensions, plus the movie-studio bridge table.

**build_dim_date()**:
- Converts release_date_raw to datetime, drops invalid dates
- Raises error if no valid dates found
- Finds min and max release dates
- Generates one row per day from min to max (increments by 1 day)
- Each row contains: date_key (YYYYMMDD integer), date, day, month, month_name, year, week_of_year, weekday_name
- Saves to `dim_date.csv`
- Returns dataframe and date_key_map (date → date_key lookup dict)

**build_dim_rating()**:
- Extracts unique MPA rating codes from mpa_rating_raw, drops nulls, sorts
- Creates rating_key starting at 0 for "UNKNOWN", then 1+ for discovered ratings
- Each row contains: rating_key, rating_code, rating_description (null for all except UNKNOWN)
- Saves to `dim_rating.csv`
- Returns dataframe and rating_key_map (rating_code → rating_key lookup dict)

**build_dim_studio_and_bridge()**:
- Iterates production_companies_raw, canonicalizes studio names using `studio_name_canonical()`
- Deduplicates by canonical name, stores original name and canonical version
- If no studios found, creates single "Unknown" studio
- Assigns studio_key starting at 1
- Saves to `dim_studio.csv`
- Builds bridge table by iterating tmdb_stg again, linking each movie's tmdb_id to its studio_keys
- Drops duplicate tmdb_id/studio_key pairs
- Saves to `bridge_movie_studio.csv`
- Returns dim_studio, bridge, and studio_key_by_canon (canonical_name → studio_key lookup dict)

**build_dim_movie()**:
- Converts release_date_raw to datetime
- Maps release dates to date_keys using date_key_map (null dates → null keys)
- Maps mpa_rating_raw to rating_keys using rating_key_map (null ratings → 0 for UNKNOWN)
- Selects relevant columns from tmdb_stg (excludes production_companies_raw)
- Coerces numeric columns to float
- Assigns movie_key starting at 1
- Saves to `dim_movie.csv`
- Returns dataframe and movie_key_map (tmdb_id → movie_key lookup dict)

All dimension tables use surrogate keys (auto-incrementing integers) as primary keys. Lookup dictionaries enable efficient foreign key mapping in subsequent stages.

In [21]:
def build_dim_date(tmdb_stg: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[date, int]]:
    release_dates = pd.to_datetime(tmdb_stg["release_date_raw"], errors="coerce")
    rd_valid = release_dates.dropna()
    if rd_valid.empty:
        raise RuntimeError("No valid release dates found to build DIM_DATE")

    min_d = rd_valid.min().date()
    max_d = rd_valid.max().date()

    days = []
    cur = min_d
    while cur <= max_d:
        date_key = int(cur.strftime("%Y%m%d"))
        days.append(
            {
                "date_key": date_key,
                "date": cur,
                "day": cur.day,
                "month": cur.month,
                "month_name": cur.strftime("%B"),
                "year": cur.year,
                "week_of_year": int(cur.strftime("%U")),
                "weekday_name": cur.strftime("%A"),
            }
        )
        cur += timedelta(days=1)

    dim_date = pd.DataFrame(days)
    dim_date.to_csv(os.path.join(DATA_DIR, "dim_date.csv"), index=False)
    date_key_map = dict(zip(dim_date["date"], dim_date["date_key"]))
    return dim_date, date_key_map


def build_dim_rating(tmdb_stg: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, int]]:
    unique = (
        tmdb_stg["mpa_rating_raw"]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    rows = [{"rating_key": 0, "rating_code": "UNKNOWN", "rating_description": "Unknown / not rated"}]
    for i, code in enumerate(unique, start=1):
        rows.append(
            {
                "rating_key": i,
                "rating_code": code,
                "rating_description": None,
            }
        )

    dim_rating = pd.DataFrame(rows)
    dim_rating.to_csv(os.path.join(DATA_DIR, "dim_rating.csv"), index=False)

    rating_key_map = dim_rating.set_index("rating_code")["rating_key"].to_dict()
    return dim_rating, rating_key_map


def build_dim_studio_and_bridge(tmdb_stg: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, int]]:
    studio_rows = []
    seen_names = set()

    for pcs in tmdb_stg["production_companies_raw"]:
        # Handle string representation from CSV
        if isinstance(pcs, str):
            try:
                pcs = ast.literal_eval(pcs)
            except Exception:
                continue
        
        if not isinstance(pcs, list):
            continue
            
        for comp in pcs:
            if not isinstance(comp, dict):
                continue
            name = comp.get("name")
            if not name:
                continue
            canon = studio_name_canonical(name)
            if canon in seen_names:
                continue
            seen_names.add(canon)
            studio_rows.append({"studio_name": name, "studio_name_canon": canon})

    dim_studio = pd.DataFrame(studio_rows)
    if dim_studio.empty:
        dim_studio = pd.DataFrame(
            [{"studio_name": "Unknown", "studio_name_canon": studio_name_canonical("Unknown")}]
        )

    dim_studio.insert(0, "studio_key", range(1, len(dim_studio) + 1))
    dim_studio.to_csv(os.path.join(DATA_DIR, "dim_studio.csv"), index=False)

    studio_key_by_canon = dim_studio.set_index("studio_name_canon")["studio_key"].to_dict()

    bridge_rows = []
    for _, row in tmdb_stg.iterrows():
        tmdb_id = row["tmdb_id"]
        pcs = row["production_companies_raw"]
        
        # Handle string representation from CSV
        if isinstance(pcs, str):
            try:
                pcs = ast.literal_eval(pcs)
            except Exception:
                continue
        
        if not isinstance(pcs, list):
            continue
            
        for comp in pcs:
            if not isinstance(comp, dict):
                continue
            name = comp.get("name")
            if not name:
                continue
            canon = studio_name_canonical(name)
            sk = studio_key_by_canon.get(canon)
            if sk is None:
                continue
            bridge_rows.append({"tmdb_id": tmdb_id, "studio_key": sk})

    bridge = pd.DataFrame(bridge_rows, columns=["tmdb_id", "studio_key"]).drop_duplicates()
    bridge.to_csv(os.path.join(DATA_DIR, "bridge_movie_studio.csv"), index=False)

    return dim_studio, bridge, studio_key_by_canon


def build_dim_movie(
    tmdb_stg: pd.DataFrame,
    date_key_map: Dict[date, int],
    rating_key_map: Dict[str, int],
) -> Tuple[pd.DataFrame, Dict[int, int]]:
    def date_to_key(dt: Any) -> Optional[int]:
        if pd.isna(dt):
            return None
        d = pd.to_datetime(dt).date()
        return date_key_map.get(d)

    def rating_to_key(code: Any) -> int:
        if pd.isna(code):
            return 0
        return rating_key_map.get(code, 0)

    tmdb_stg["release_date"] = pd.to_datetime(tmdb_stg["release_date_raw"], errors="coerce")
    tmdb_stg["release_date_key"] = tmdb_stg["release_date"].apply(date_to_key)
    tmdb_stg["rating_key"] = tmdb_stg["mpa_rating_raw"].apply(rating_to_key)

    dim_movie = tmdb_stg[
        [
            "tmdb_id",
            "imdb_id",
            "title",
            "original_title",
            "release_date_key",
            "original_language",
            "runtime_minutes",
            "status",
            "budget_usd",
            "primary_genre",
            "secondary_genre",
            "collection_name",
            "rating_key",
            "popularity",
            "vote_average",
            "vote_count",
            "production_companies",
            "director_popularity",
            "cast_popularity",
            "max_cast_popularity",
        ]
    ].copy()

    for col in ["director_popularity", "cast_popularity", "budget_usd", "runtime_minutes", "popularity"]:
        dim_movie[col] = pd.to_numeric(dim_movie[col], errors="coerce").astype(float)

    dim_movie.insert(0, "movie_key", range(1, len(dim_movie) + 1))
    dim_movie.to_csv(os.path.join(DATA_DIR, "dim_movie.csv"), index=False)

    movie_key_map = dim_movie.set_index("tmdb_id")["movie_key"].to_dict()
    return dim_movie, movie_key_map


## Section 7 - Fact: Opening Weekend

This section builds the fact table containing opening weekend box office revenue with foreign key references to the movie dimension.

**Process**:
1. Copies `open_stg` dataframe
2. Maps tmdb_id to movie_key using `movie_key_map` lookup dictionary
3. Selects fact columns: movie_key, opening_weekend_gross_usd, opening_source, opening_source_url
4. Drops rows with null movie_key or null opening_weekend_gross_usd
5. Filters out rows where opening_weekend_gross_usd ≤ 0
6. Coerces opening_weekend_gross_usd to float
7. Inserts fact_id (surrogate key starting at 1)
8. Saves to `fact_opening_weekend.csv`
9. Prints summary: row count, min and max opening weekend revenue

**Output**: Fact table with one row per movie that has valid opening weekend data. Each row links to dim_movie via movie_key foreign key and includes source attribution for data lineage.

In [22]:
def build_fact_opening(
    open_stg: pd.DataFrame,
    movie_key_map: Dict[int, int],
) -> pd.DataFrame:
    merged = open_stg.copy()
    merged["movie_key"] = merged["tmdb_id"].map(movie_key_map)

    fact = merged[
        [
            "movie_key",
            "opening_weekend_gross_usd",
            "opening_source",
            "opening_source_url",
        ]
    ].copy()

    fact = fact.dropna(subset=["movie_key", "opening_weekend_gross_usd"])
    fact = fact[fact["opening_weekend_gross_usd"] > 0]

    fact["opening_weekend_gross_usd"] = pd.to_numeric(
        fact["opening_weekend_gross_usd"], errors="coerce"
    ).astype(float)

    fact.insert(0, "fact_id", range(1, len(fact) + 1))
    fact.to_csv(os.path.join(DATA_DIR, "fact_opening_weekend.csv"), index=False)

    print("FACT_OPENING summary:")
    print(f"  rows: {len(fact)}")
    if not fact.empty:
        print(f"  min opening_weekend_gross_usd: {fact['opening_weekend_gross_usd'].min():,.0f}")
        print(f"  max opening_weekend_gross_usd: {fact['opening_weekend_gross_usd'].max():,.0f}")

    return fact

## Section 8 - Flat Raw Dataset

This section joins all dimensional and fact tables into a single denormalized dataset for easy consumption and exploratory analysis.

**Process**:
1. Starts with `dim_movie` as base
2. Left joins `dim_date` on release_date_key = date_key (adds date attributes: day, month, year, weekday_name, etc.)
3. Left joins `dim_rating` on rating_key (adds rating_code and rating_description)
4. Left joins `fact_opening` on movie_key (adds opening_weekend_gross_usd, opening_source, opening_source_url)
5. Handles many-to-many studio relationships:
   - Joins bridge_movie_studio with dim_studio to get studio names
   - Groups by tmdb_id and aggregates studio names into comma-separated string
   - Filters empty/whitespace names, deduplicates, sorts alphabetically
   - Merges aggregated studios_from_dim column back to main dataframe
6. Coerces numeric columns to float: director_popularity, cast_popularity, budget_usd, runtime_minutes, popularity, opening_weekend_gross_usd
7. Saves to `raw_movies_all.csv`
8. Prints save confirmation with row count

**Output**: Fully denormalized dataset with one row per movie. All dimension attributes and fact metrics are joined into a single flat table. Movies without opening weekend data will have nulls in opening_weekend_gross_usd column.

In [23]:
def _safe_datetime(val: Any) -> pd.Timestamp:
    """Parse various date representations into a pandas Timestamp (NaT on failure)."""
    return pd.to_datetime(val, errors="coerce")


# ============================================================
# DISK-BACKED CACHING FOR PERSON CREDITS & OPENING WEEKENDS
# ============================================================

def _load_person_cache() -> Dict[int, Dict[str, Any]]:
    """Load person credits from disk cache."""
    cache_path = os.path.join(PERSON_CACHE_DIR, "person_credits.parquet")
    if os.path.exists(cache_path) and not REFRESH_CACHE:
        try:
            df = pd.read_parquet(cache_path)
            return {int(row["person_id"]): eval(row["credits"]) for _, row in df.iterrows()}
        except Exception:
            pass
    return {}


def _save_person_cache(cache: Dict[int, Dict[str, Any]]) -> None:
    """Save person credits to disk cache."""
    cache_path = os.path.join(PERSON_CACHE_DIR, "person_credits.parquet")
    try:
        records = [{"person_id": pid, "credits": str(credits)} for pid, credits in cache.items()]
        pd.DataFrame(records).to_parquet(cache_path, index=False)
    except Exception as e:
        print(f"[CACHE] Warning: failed to save person cache: {e}")


def _load_opening_cache() -> Dict[int, float]:
    """Load opening weekends from disk cache."""
    cache_path = os.path.join(OPENING_CACHE_DIR, "opening_cache.parquet")
    if os.path.exists(cache_path) and not REFRESH_CACHE:
        try:
            df = pd.read_parquet(cache_path)
            return dict(zip(df["tmdb_id"].astype(int), df["opening_weekend"].astype(float)))
        except Exception:
            pass
    return {}


def _save_opening_cache(cache: Dict[int, float]) -> None:
    """Save opening weekends to disk cache."""
    cache_path = os.path.join(OPENING_CACHE_DIR, "opening_cache.parquet")
    try:
        records = [{"tmdb_id": mid, "opening_weekend": val} for mid, val in cache.items() if not np.isnan(val)]
        pd.DataFrame(records).to_parquet(cache_path, index=False)
    except Exception as e:
        print(f"[CACHE] Warning: failed to save opening cache: {e}")


# ============================================================
# OPTIMIZED STAR-POWER FEATURE COMPUTATION (BATCH MODE)
# ============================================================

def append_star_power_columns(df: pd.DataFrame, open_stg: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Compute director/lead-actor prior-history features and append them to df.

    ALWAYS computes:
      - director_has_prior, director_prior_count
      - lead_actor_has_prior, lead_actor_prior_count
    
    When FETCH_RECENT_OPENING_FEATURES is True, also computes:
      - director_recent_opening, lead_actor_recent_opening
    
    When False, sets recent-opening columns to NaN (skips expensive scraping).
    """

    if df.empty:
        return df

    df = df.copy()

    # ── Step 0: Always load person credit cache (needed for prior history) ──
    person_credit_cache = _load_person_cache()

    # ── Step 1: Load opening cache only if toggle is ON ──
    opening_cache: Dict[int, float] = {}
    if FETCH_RECENT_OPENING_FEATURES:
        opening_cache = _load_opening_cache()
        
        # Tier 1: Dataset openings
        if open_stg is not None and not open_stg.empty:
            try:
                tmp = open_stg[["tmdb_id", "opening_weekend_gross_usd"]].dropna()
                tmp["tmdb_id"] = pd.to_numeric(tmp["tmdb_id"], errors="coerce").astype(int)
                tmp["opening_weekend_gross_usd"] = pd.to_numeric(
                    tmp["opening_weekend_gross_usd"], errors="coerce"
                ).astype(float)
                opening_cache.update(dict(zip(tmp["tmdb_id"], tmp["opening_weekend_gross_usd"])))
            except Exception:
                pass

    # ── Step 2: Extract unique people from dataset ──
    print("[STAR-POWER] Extracting unique people from dataset...")
    unique_directors: Dict[int, Dict[str, Any]] = {}
    unique_leads: Dict[int, Dict[str, Any]] = {}
    movie_people_map: Dict[int, Dict[str, Any]] = {}  # tmdb_id → {directors: [...], lead: {...}, release_dt: ...}

    for _, row in df.iterrows():
        tmdb_id = row.get("tmdb_id")
        if tmdb_id is None:
            continue
        
        release_dt = _safe_datetime(row.get("date") or row.get("release_date") or row.get("release_date_raw"))
        
        try:
            detail = tmdb_movie_full(int(tmdb_id))
        except Exception:
            continue
        
        credits = (detail or {}).get("credits") or {}
        crew = credits.get("crew") or []
        cast = credits.get("cast") or []

        directors = [c for c in crew if c.get("job") == "Director"]
        lead = None
        if isinstance(cast, list) and cast:
            lead = next((c for c in cast if c.get("order") == 0), cast[0])

        movie_people_map[int(tmdb_id)] = {
            "directors": directors,
            "lead": lead,
            "release_dt": release_dt
        }

        for d in directors:
            pid = d.get("id")
            if pid is not None:
                unique_directors[int(pid)] = d

        if lead is not None:
            pid = lead.get("id")
            if pid is not None:
                unique_leads[int(pid)] = lead

    print(f"[STAR-POWER] Found {len(unique_directors)} unique directors, {len(unique_leads)} unique lead actors")

    # ── Step 3: Fetch filmographies for unique people (ALWAYS, needed for prior counts) ──
    def _fetch_person_credits(person_id: int) -> Dict[str, Any]:
        if person_id in person_credit_cache:
            return person_credit_cache[person_id]
        try:
            time.sleep(0.05)  # Rate limiting
            data = tmdb_get(f"/person/{int(person_id)}/movie_credits")
            person_credit_cache[person_id] = data
            return data
        except Exception:
            person_credit_cache[person_id] = {}
            return {}

    print("[STAR-POWER] Fetching filmographies (with disk caching)...")
    all_person_ids = set(unique_directors.keys()) | set(unique_leads.keys())
    
    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, 5)) as executor:
        futures = {executor.submit(_fetch_person_credits, pid): pid for pid in all_person_ids}
        for future in tqdm(as_completed(futures), total=len(all_person_ids), desc="Fetching credits", unit="person"):
            try:
                future.result()
            except Exception:
                pass

    # Save updated person cache
    _save_person_cache(person_credit_cache)

    # ── Step 4: Precompute prior metrics for each unique person ──
    def _compute_person_prior_history(person_id: int, role: str, reference_dates: List[pd.Timestamp]) -> Dict[pd.Timestamp, Tuple[int, int]]:
        """Return dict: reference_date → (has_prior, prior_count)
        
        This computes ONLY prior history, regardless of FETCH_RECENT_OPENING_FEATURES.
        """
        credits = person_credit_cache.get(person_id, {})
        
        if role == "director":
            filmography = [c for c in credits.get("crew", []) if c.get("job") == "Director"]
        else:
            filmography = credits.get("cast", [])

        if not isinstance(filmography, list):
            filmography = []

        # Parse all prior films
        prior_films: List[Tuple[pd.Timestamp, int]] = []
        for entry in filmography:
            rd = _safe_datetime(entry.get("release_date"))
            if pd.isna(rd):
                continue
            mid = entry.get("id")
            if mid is None:
                continue
            try:
                prior_films.append((rd, int(mid)))
            except Exception:
                continue

        prior_films.sort(key=lambda x: x[0])

        # Compute metrics for each reference date
        results = {}
        for ref_dt in reference_dates:
            if pd.isna(ref_dt):
                results[ref_dt] = (0, 0)
                continue

            # Films strictly before ref_dt
            earlier = [f for f in prior_films if f[0] < ref_dt]
            prior_count = len(earlier)
            has_prior = int(prior_count > 0)

            results[ref_dt] = (has_prior, prior_count)

        return results

    def _compute_person_recent_opening(person_id: int, role: str, reference_dates: List[pd.Timestamp]) -> Dict[pd.Timestamp, float]:
        """Return dict: reference_date → recent_opening (in USD)
        
        This ONLY computes if FETCH_RECENT_OPENING_FEATURES is True.
        Requires filmography to already be cached.
        """
        if not FETCH_RECENT_OPENING_FEATURES:
            return {ref_dt: np.nan for ref_dt in reference_dates}

        credits = person_credit_cache.get(person_id, {})
        
        if role == "director":
            filmography = [c for c in credits.get("crew", []) if c.get("job") == "Director"]
        else:
            filmography = credits.get("cast", [])

        if not isinstance(filmography, list):
            filmography = []

        # Parse all prior films with metadata needed for scraping
        prior_films: List[Tuple[pd.Timestamp, int, str]] = []
        for entry in filmography:
            rd = _safe_datetime(entry.get("release_date"))
            if pd.isna(rd):
                continue
            mid = entry.get("id")
            title = entry.get("title") or entry.get("original_title") or entry.get("name")
            if mid is None or title is None:
                continue
            try:
                prior_films.append((rd, int(mid), str(title)))
            except Exception:
                continue

        prior_films.sort(key=lambda x: x[0])

        # Compute metrics for each reference date
        results = {}
        for ref_dt in reference_dates:
            if pd.isna(ref_dt):
                results[ref_dt] = np.nan
                continue

            # Films strictly before ref_dt
            earlier = [f for f in prior_films if f[0] < ref_dt]
            if not earlier:
                results[ref_dt] = np.nan
                continue

            # Find most recent opening (walk backward)
            recent_opening = np.nan
            for rel_dt, pmid, ptitle in reversed(earlier):
                # Layered lookup: cache → scrape
                if pmid in opening_cache:
                    cand = opening_cache[pmid]
                else:
                    # Scrape only if not in cache
                    cand = _scrape_opening(pmid, ptitle, rel_dt)
                    if not np.isnan(cand) and cand >= MIN_OPENING_WEEKEND_USD:
                        opening_cache[pmid] = cand
                
                if not np.isnan(cand) and cand >= MIN_OPENING_WEEKEND_USD:
                    recent_opening = cand
                    break

            results[ref_dt] = recent_opening

        return results

    def _scrape_opening(tmdb_id: int, title: str, release_dt: pd.Timestamp) -> float:
        """Scrape opening weekend from BOM → The Numbers (last resort)."""
        try:
            detail = tmdb_movie_full(int(tmdb_id))
        except Exception:
            return np.nan

        imdb_id = (detail.get("external_ids") or {}).get("imdb_id")
        release_year = safe_year(detail.get("release_date"))
        if pd.notna(release_dt):
            try:
                release_year = int(release_dt.year)
            except Exception:
                pass

        val = np.nan

        # Try BOM
        if isinstance(imdb_id, str) and imdb_id.startswith("tt"):
            url_bom = bom_url_from_imdb(imdb_id)
            html_bom = fetch_html(url_bom)
            val_bom = parse_bom_opening(html_bom)
            if not np.isnan(val_bom):
                val = float(val_bom)

        # Fallback: The Numbers
        if (np.isnan(val)) and release_year and title:
            url_nums = numbers_url_from_title_year(str(title), int(release_year))
            html_nums = fetch_html(url_nums)
            val_nums = parse_numbers_opening(html_nums)
            if not np.isnan(val_nums):
                val = float(val_nums)

        if np.isnan(val) or val < MIN_OPENING_WEEKEND_USD:
            return np.nan
        return val

    print("[STAR-POWER] Precomputing prior histories for unique people...")
    # Group reference dates by person
    dir_ref_dates: Dict[int, List[pd.Timestamp]] = {}
    lead_ref_dates: Dict[int, List[pd.Timestamp]] = {}

    for movie_data in movie_people_map.values():
        ref_dt = movie_data["release_dt"]
        for d in movie_data["directors"]:
            pid = d.get("id")
            if pid is not None:
                dir_ref_dates.setdefault(int(pid), []).append(ref_dt)
        lead = movie_data["lead"]
        if lead is not None:
            pid = lead.get("id")
            if pid is not None:
                lead_ref_dates.setdefault(int(pid), []).append(ref_dt)

    # Compute prior history for all directors and leads (ALWAYS)
    director_prior_histories: Dict[int, Dict[pd.Timestamp, Tuple[int, int]]] = {}
    lead_prior_histories: Dict[int, Dict[pd.Timestamp, Tuple[int, int]]] = {}

    for pid in tqdm(unique_directors.keys(), desc="Directors (prior history)", unit="person"):
        ref_dates = dir_ref_dates.get(pid, [])
        director_prior_histories[pid] = _compute_person_prior_history(pid, "director", ref_dates)

    for pid in tqdm(unique_leads.keys(), desc="Lead actors (prior history)", unit="person"):
        ref_dates = lead_ref_dates.get(pid, [])
        lead_prior_histories[pid] = _compute_person_prior_history(pid, "actor", ref_dates)

    # Compute recent opening history ONLY if toggle is ON
    director_recent_histories: Dict[int, Dict[pd.Timestamp, float]] = {}
    lead_recent_histories: Dict[int, Dict[pd.Timestamp, float]] = {}

    if FETCH_RECENT_OPENING_FEATURES:
        print("[STAR-POWER] Precomputing recent opening weekends for unique people...")
        for pid in tqdm(unique_directors.keys(), desc="Directors (recent opening)", unit="person"):
            ref_dates = dir_ref_dates.get(pid, [])
            director_recent_histories[pid] = _compute_person_recent_opening(pid, "director", ref_dates)

        for pid in tqdm(unique_leads.keys(), desc="Lead actors (recent opening)", unit="person"):
            ref_dates = lead_ref_dates.get(pid, [])
            lead_recent_histories[pid] = _compute_person_recent_opening(pid, "actor", ref_dates)

        # Save updated opening cache
        _save_opening_cache(opening_cache)

    # ── Step 5: Merge precomputed metrics back into dataframe ──
    print("[STAR-POWER] Merging precomputed metrics into dataframe...")
    director_has_prior: List[int] = []
    director_prior_count: List[int] = []
    director_recent_opening: List[float] = []

    lead_has_prior: List[int] = []
    lead_prior_count: List[int] = []
    lead_recent_opening: List[float] = []

    for _, row in df.iterrows():
        tmdb_id = row.get("tmdb_id")
        movie_data = movie_people_map.get(int(tmdb_id) if tmdb_id is not None else -1)
        
        if movie_data is None:
            director_has_prior.append(0)
            director_prior_count.append(0)
            director_recent_opening.append(np.nan)
            lead_has_prior.append(0)
            lead_prior_count.append(0)
            lead_recent_opening.append(np.nan)
            continue

        directors = movie_data["directors"]
        lead = movie_data["lead"]
        ref_dt = movie_data["release_dt"]

        # Directors (MAX aggregation for prior history)
        if not directors:
            director_has_prior.append(0)
            director_prior_count.append(0)
            director_recent_opening.append(np.nan)
        else:
            prior_metrics = []
            recent_metrics = []
            for d in directors:
                pid = d.get("id")
                if pid is not None and int(pid) in director_prior_histories:
                    prior_hist = director_prior_histories[int(pid)]
                    prior_metrics.append(prior_hist.get(ref_dt, (0, 0)))
                if pid is not None and int(pid) in director_recent_histories:
                    recent_hist = director_recent_histories[int(pid)]
                    recent_metrics.append(recent_hist.get(ref_dt, np.nan))
            
            if prior_metrics:
                director_has_prior.append(int(any(m[0] for m in prior_metrics)))
                director_prior_count.append(int(max((m[1] for m in prior_metrics), default=0)))
            else:
                director_has_prior.append(0)
                director_prior_count.append(0)

            if recent_metrics:
                recent_vals = [m for m in recent_metrics if not np.isnan(m)]
                director_recent_opening.append(float(max(recent_vals)) if recent_vals else np.nan)
            else:
                director_recent_opening.append(np.nan)

        # Lead actor (MAX aggregation for prior history)
        if lead is None:
            lead_has_prior.append(0)
            lead_prior_count.append(0)
            lead_recent_opening.append(np.nan)
        else:
            pid = lead.get("id")
            has_p, cnt_p = (0, 0)
            recent_o = np.nan
            
            if pid is not None and int(pid) in lead_prior_histories:
                prior_hist = lead_prior_histories[int(pid)]
                prior_metrics = prior_hist.get(ref_dt, (0, 0))
                has_p, cnt_p = prior_metrics
            
            if pid is not None and int(pid) in lead_recent_histories:
                recent_hist = lead_recent_histories[int(pid)]
                recent_o = recent_hist.get(ref_dt, np.nan)
            
            lead_has_prior.append(int(has_p))
            lead_prior_count.append(int(cnt_p))
            lead_recent_opening.append(recent_o)

    df["director_has_prior"] = director_has_prior
    df["director_prior_count"] = director_prior_count
    df["director_recent_opening"] = director_recent_opening

    df["lead_actor_has_prior"] = lead_has_prior
    df["lead_actor_prior_count"] = lead_prior_count
    df["lead_actor_recent_opening"] = lead_recent_opening

    print("[STAR-POWER] Feature computation complete.")
    return df


def enforce_star_power_validations(df: pd.DataFrame) -> pd.DataFrame:
    """Apply strict consistency rules for star-power features on preprocessed datasets.
    
    ONLY runs when FETCH_RECENT_OPENING_FEATURES is True, since the validations
    assume recent-opening columns exist and contain valid data.
    
    When toggle is False, this function should not be called and rows should not
    be dropped due to missing recent-opening values.
    """
    if not FETCH_RECENT_OPENING_FEATURES:
        # Skip validation when toggle is OFF to avoid dropping rows
        return df
    
    if df.empty:
        return df

    df = df.copy()

    def _apply(prefix: str, frame: pd.DataFrame) -> pd.DataFrame:
        has_col = f"{prefix}_has_prior"
        cnt_col = f"{prefix}_prior_count"
        recent_col = f"{prefix}_recent_opening"
        if not {has_col, cnt_col, recent_col}.issubset(frame.columns):
            return frame

        conflict = (frame[has_col] == 0) & (
            (frame[cnt_col] > 0) | (frame[recent_col] > 0)
        )
        if conflict.any():
            frame.loc[conflict, [cnt_col, recent_col]] = np.nan

        missing_count = (frame[has_col] == 1) & ((frame[cnt_col].isna()) | (frame[cnt_col] == 0))
        frame = frame[~missing_count]

        missing_recent = (frame[has_col] == 1) & (frame[cnt_col] > 0) & (frame[recent_col].isna())
        frame = frame[~missing_recent]

        return frame

    df = _apply("director", df)
    df = _apply("lead_actor", df)
    return df


In [24]:
def build_flat_raw_dataset(
    dim_movie: pd.DataFrame,
    dim_date: pd.DataFrame,
    dim_rating: pd.DataFrame,
    dim_studio: pd.DataFrame,
    bridge_movie_studio: pd.DataFrame,
    fact_opening: pd.DataFrame,
    open_stg: pd.DataFrame,
) -> pd.DataFrame:
    df = dim_movie.copy()

    df = df.merge(
        dim_date,
        left_on="release_date_key",
        right_on="date_key",
        how="left",
        suffixes=("", "_date"),
    )

    df = df.merge(
        dim_rating,
        on="rating_key",
        how="left",
        suffixes=("", "_rating"),
    )

    fact_cols = [
        "movie_key",
        "opening_weekend_gross_usd",
        "opening_source",
        "opening_source_url",
    ]
    df = df.merge(
        fact_opening[fact_cols],
        on="movie_key",
        how="left",
    )

    studios = bridge_movie_studio.merge(
        dim_studio[["studio_key", "studio_name"]],
        on="studio_key",
        how="left",
    )

    def agg_studios(s: pd.Series) -> Optional[str]:
        names = [x for x in s if isinstance(x, str) and x.strip()]
        if not names:
            return None
        names = sorted(set(names))
        return ", ".join(names)

    studios_grouped = (
        studios.groupby("tmdb_id")["studio_name"]
        .apply(agg_studios)
        .reset_index()
        .rename(columns={"studio_name": "studios_from_dim"})
    )

    df = df.merge(studios_grouped, on="tmdb_id", how="left")

    df = append_star_power_columns(df, open_stg=open_stg)

    for col in ["director_popularity", "cast_popularity", "max_cast_popularity", "budget_usd", "runtime_minutes", "popularity"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    if "opening_weekend_gross_usd" in df.columns:
        df["opening_weekend_gross_usd"] = pd.to_numeric(
            df["opening_weekend_gross_usd"], errors="coerce"
        ).astype(float)

    out_path = os.path.join(DATA_DIR, "raw_movies_all.csv")
    df.to_csv(out_path, index=False)
    print(f"Raw denormalized movies CSV saved → {out_path} (rows: {len(df)})")

    return df

## Section 9 - Raw Avatar 3 Only

This section fetches Avatar 3 metadata from TMDB and saves it as a separate single-row raw dataset for prediction purposes.

**Process**:
1. Fetches full Avatar 3 details using `tmdb_movie_full(AVATAR3_TMDB_ID)`
2. If fetch fails, prints warning and returns empty dataframe
3. Extracts same metadata structure as `build_tmdb_staging()`:
   - Basic info: tmdb_id, imdb_id, title, original_title, release_date_raw, original_language, runtime_minutes, status
   - Financial: budget_usd
   - Metrics: popularity, vote_average, vote_count
   - Genres: primary_genre (first), secondary_genre (second)
   - Collection: collection_name (franchise membership)
   - Rating: mpa_rating_raw (extracted using `extract_mpa_rating_from_movie_dict()`)
   - Studios: production_companies_raw (list of dicts), production_companies (comma-separated string)
   - Popularity metrics: director_popularity (mean of all directors), cast_popularity (mean of top 5 cast)
4. Creates single-row dataframe from extracted metadata
5. Coerces 7 numeric columns to float: budget_usd, popularity, vote_average, vote_count, runtime_minutes, director_popularity, cast_popularity
6. Saves to `raw_avatar3_only.csv`
7. Prints save confirmation

**Output**: Single-row raw TMDB snapshot of Avatar 3. This serves as the prediction target and is kept separate from training data to prevent data leakage.

In [25]:
def build_raw_avatar3_only(open_stg: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    try:
        d = tmdb_movie_full(AVATAR3_TMDB_ID)
    except Exception as e:
        print(f"Could not fetch Avatar 3 details from TMDB: {e}")
        return pd.DataFrame()

    genres = d.get("genres") or []
    genre_names = [g.get("name") for g in genres if g.get("name")]
    primary_genre = genre_names[0] if genre_names else None
    secondary_genre = genre_names[1] if len(genre_names) > 1 else None

    belongs = d.get("belongs_to_collection")
    collection_name = belongs.get("name") if isinstance(belongs, dict) else None

    mpa_raw = extract_mpa_rating_from_movie_dict(d)

    prod_raw = d.get("production_companies") or []
    prod_names = [pc.get("name") for pc in prod_raw if pc.get("name")]
    prod_names_str = ", ".join(prod_names) if prod_names else None

    credits = d.get("credits") or {}
    crew = credits.get("crew") or []
    cast = credits.get("cast") or []

    directors = [c for c in crew if c.get("job") == "Director"]
    director_popularity = np.nanmean(
        [x.get("popularity", np.nan) for x in directors]
    ) if directors else np.nan

    top_cast = cast[:5]
    cast_popularity = np.nanmean(
        [x.get("popularity", np.nan) for x in top_cast]
    ) if top_cast else np.nan
    max_cast_popularity = np.nanmax(
        [x.get("popularity", np.nan) for x in top_cast]
    ) if top_cast else np.nan

    row = {
        "tmdb_id": d.get("id"),
        "imdb_id": (d.get("external_ids") or {}).get("imdb_id"),
        "title": d.get("title"),
        "original_title": d.get("original_title"),
        "release_date_raw": d.get("release_date"),
        "original_language": d.get("original_language"),
        "runtime_minutes": d.get("runtime"),
        "status": d.get("status"),
        "budget_usd": d.get("budget"),
        "popularity": d.get("popularity"),
        "vote_average": d.get("vote_average"),
        "vote_count": d.get("vote_count"),
        "primary_genre": primary_genre,
        "secondary_genre": secondary_genre,
        "collection_name": collection_name,
        "mpa_rating_raw": mpa_raw,
        "production_companies_raw": prod_raw,
        "production_companies": prod_names_str,
        "director_popularity": director_popularity,
        "cast_popularity": cast_popularity,
        "max_cast_popularity": max_cast_popularity,
    }

    df = pd.DataFrame([row])

    numeric_cols = [
        "budget_usd",
        "popularity",
        "vote_average",
        "vote_count",
        "runtime_minutes",
        "director_popularity",
        "cast_popularity",
        "max_cast_popularity",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    df["date"] = pd.to_datetime(df["release_date_raw"], errors="coerce")
    df = append_star_power_columns(df, open_stg=open_stg)

    out_path = os.path.join(DATA_DIR, "raw_avatar3_only.csv")
    df.to_csv(out_path, index=False)
    print(f" Saved Avatar 3 raw TMDB snapshot → {out_path}")
    return df

## Section 10 - Feature Engineering Helpers

This section provides helper functions for computing collection-based features and validating engineered feature datasets.

**compute_collection_past_open_avg()**:
- Adds `collection_past_open_avg` column initialized to 0.0
- Determines release date from available columns (`release_date`, `release_date_raw`, or constructed from `release_year`/`release_month`)
- Filters movies that have both `collection_name` and opening weekend data
- If no movies meet criteria, returns dataframe with 0.0 values
- For each collection:
  - Fetches full franchise history from TMDB's `/collection/{id}` endpoint
  - For franchise parts not in dataset, scrapes opening weekend from BOM/The Numbers
  - Combines dataset movies with discovered franchise parts
  - Sorts by release date (chronological order within each franchise)
  - Computes expanding mean of past opening weekends using shift(1) to exclude current movie
  - First movie in each franchise gets NaN → filled with 0.0
- Maps computed values back to original dataframe indices
- Returns dataframe with `collection_past_open_avg` populated for franchise films

Please check **Note on `collection_past_open_avg`** in Section 11

**validate_feature_frame()**:
- Prints validation summary with row count
- Checks required columns for missing values: `title`, `release_year`, `log_budget`, `runtime_minutes`
- Validates binary flag columns are strictly 0 or 1: `is_summer`, `is_holiday`, `is_q1_dump_window`, `is_spring_dead_zone`, `is_awards_season`, `is_major_studio`, `is_english`, `is_collection`, `director_has_prior`, `lead_actor_has_prior`
- If `is_training=True`, validates `opening_weekend` column has no missing values and all values > 0
- Prints validation results with checkmarks (✅) for passing checks and warnings (⚠️) for failures
- Does not return anything (validation only, no modifications)

These helpers ensure data quality and compute temporal franchise features that prevent data leakage by only using information available at each movie's release date.

In [26]:
def compute_collection_past_open_avg(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute collection_past_open_avg using FULL franchise history,
    not just movies inside the training year window.

    For each movie in a collection:
        mean(opening_weekend of all earlier movies in that collection),
    where "earlier" is defined by actual release date.

    This version:
      - Uses df's existing movies (with opening_weekend_gross_usd) for the collection.
      - Calls TMDB's /collection/{id} to discover ALL franchise entries.
      - For parts NOT in df, fetches opening weekend via BOM + The Numbers,
        assuming parse_bom_opening / parse_numbers_opening may return either:
            * float
            * or (float, date_str) tuple.
      - Ensures no leakage: only strictly earlier release dates are used.
    """
    df = df.copy()

    # Choose the opening-weekend column in the current schema
    if "opening_weekend_gross_usd" in df.columns:
        open_col = "opening_weekend_gross_usd"
    elif "opening_weekend" in df.columns:
        open_col = "opening_weekend"
    else:
        print("[WARN] No opening weekend column found in df.")
        df["collection_past_open_avg"] = 0.0
        return df

    df["collection_past_open_avg"] = 0.0

    required_cols = ["tmdb_id", "title", "collection_name", open_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"[WARN] compute_collection_past_open_avg missing columns: {missing}")
        return df

    if "release_date" in df.columns:
        df["release_date_dt"] = pd.to_datetime(df["release_date"], errors="coerce")
    elif "release_date_raw" in df.columns:
        df["release_date_dt"] = pd.to_datetime(df["release_date_raw"], errors="coerce")
    elif {"release_year", "release_month"}.issubset(df.columns):
        df["release_date_dt"] = pd.to_datetime(
            df["release_year"].astype(int).astype(str)
            + "-"
            + df["release_month"].fillna(7).astype(int).astype(str)
            + "-15",
            errors="coerce",
        )
    else:
        df["release_date_dt"] = pd.to_datetime(
            df["release_year"].astype(int).astype(str) + "-07-01",
            errors="coerce",
        )

    collections = (
        df["collection_name"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )
    if not collections:
        return df

    collection_parts_cache: Dict[int, List[Dict[str, Any]]] = {}
    part_opening_cache: Dict[int, float] = {}
    existing_tmdb_ids = set(df["tmdb_id"].dropna().astype(int).tolist())

    def _opening_from_parser_result(res: Any) -> float:
        if isinstance(res, (tuple, list)) and len(res) >= 1:
            return float(res[0]) if res[0] is not None else np.nan
        try:
            return float(res)
        except Exception:
            return np.nan

    for cname in tqdm(collections, desc="Processing franchise collections", unit="collection"):
        sub = df[df["collection_name"] == cname].copy()
        if sub.empty:
            continue

        example_tmdb_id = int(sub["tmdb_id"].iloc[0])

        try:
            movie_detail = tmdb_movie_full(example_tmdb_id)
        except Exception as e:
            print(f"[COLL] TMDB error fetching {example_tmdb_id}: {e}")
            continue

        belongs = movie_detail.get("belongs_to_collection")
        if not isinstance(belongs, dict) or "id" not in belongs:
            continue

        collection_id = belongs["id"]

        if collection_id not in collection_parts_cache:
            try:
                coll_json = tmdb_get(f"/collection/{collection_id}")
                collection_parts_cache[collection_id] = coll_json.get("parts", []) or []
            except Exception as e:
                print(f"[COLL] Error fetching collection {collection_id}: {e}")
                collection_parts_cache[collection_id] = []

        parts = collection_parts_cache[collection_id]

        history_rows: List[Dict[str, Any]] = []

        history_rows.extend(
            sub[["tmdb_id", "title", "release_date_dt", open_col]]
            .rename(columns={open_col: "opening_weekend"})
            .to_dict(orient="records")
        )

        for p in parts:
            part_tmdb_id = p.get("id")
            if part_tmdb_id is None:
                continue
            try:
                part_tmdb_id = int(part_tmdb_id)
            except Exception:
                continue

            if part_tmdb_id in existing_tmdb_ids:
                continue

            part_title = p.get("title") or p.get("name")
            part_release = p.get("release_date")
            part_release_dt = pd.to_datetime(part_release, errors="coerce")
            if pd.isna(part_release_dt):
                continue

            if part_tmdb_id not in part_opening_cache:
                opening_val = np.nan

                try:
                    detail_part = tmdb_movie_full(part_tmdb_id)
                except Exception as e:
                    print(f"[COLL] TMDB detail error for {part_tmdb_id}: {e}")
                    part_opening_cache[part_tmdb_id] = np.nan
                    continue

                imdb_id = (detail_part.get("external_ids") or {}).get("imdb_id")

                if isinstance(imdb_id, str) and imdb_id.startswith("tt"):
                    url_bom = bom_url_from_imdb(imdb_id)
                    html_bom = fetch_html(url_bom)
                    res_bom = parse_bom_opening(html_bom)
                    opening_val = _opening_from_parser_result(res_bom)

                if np.isnan(opening_val) or opening_val < MIN_OPENING_WEEKEND_USD:
                    continue
                    year_val = safe_year(detail_part.get("release_date"))
                    if not np.isnan(year_val):
                        url_nums = numbers_url_from_title_year(
                            str(part_title), int(year_val)
                        )
                        html_nums = fetch_html(url_nums)
                        res_nums = parse_numbers_opening(html_nums)
                        opening_val = _opening_from_parser_result(res_nums)

                part_opening_cache[part_tmdb_id] = opening_val

            opening_val = part_opening_cache.get(part_tmdb_id, np.nan)
            if np.isnan(opening_val):
                continue

            history_rows.append(
                {
                    "tmdb_id": part_tmdb_id,
                    "title": part_title,
                    "release_date_dt": part_release_dt,
                    "opening_weekend": opening_val,
                }
            )

        if not history_rows:
            continue

        history_df = pd.DataFrame(history_rows).dropna(
            subset=["release_date_dt", "opening_weekend"]
        )
        if history_df.empty:
            continue

        history_df = history_df.sort_values("release_date_dt").reset_index(drop=True)
        history_df["collection_past_open_avg"] = (
            history_df["opening_weekend"].shift(1).expanding().mean()
        )

        past_map = history_df.set_index("tmdb_id")["collection_past_open_avg"].to_dict()
        mask = df["collection_name"] == cname

        df.loc[mask, "collection_past_open_avg"] = (
            df.loc[mask, "tmdb_id"].map(past_map).fillna(0.0)
        )

    return df


def validate_feature_frame(df: pd.DataFrame, is_training: bool = True) -> None:
    print("\n[VALIDATION] Feature frame summary:")
    print("  rows:", len(df))

    required_cols = [
        "title",
        "release_year",
        "log_budget",
        "runtime_minutes",
    ]
    missing_required = [c for c in required_cols if df[c].isna().any()]
    if missing_required:
        print("  ⚠️ Columns with missing values:", missing_required)
    else:
        print("  ✅ No missing values in required core columns.")

    flag_cols = [
        "is_summer",
        "is_holiday",
        "is_q1_dump_window",
        "is_spring_dead_zone",
        "is_awards_season",
        "is_major_studio",
        "is_english",
        "is_collection",
        "director_has_prior",
        "lead_actor_has_prior",
    ]
    for c in flag_cols:
        bad = ~df[c].isin([0, 1])
        if bad.any():
            print(f"  ⚠️ Column {c} has non-binary values in {bad.sum()} rows.")
        else:
            print(f"  ✅ Column {c} is binary (0/1).")

    if is_training:
        if df["opening_weekend"].isna().any():
            print("  ⚠️ Training frame has missing opening_weekend values.")
        if (df["opening_weekend"] <= 0).any():
            print("  ⚠️ Training frame has non-positive opening_weekend values.")
        else:
            print("  ✅ Training targets (opening_weekend) look valid.")

    print("[VALIDATION] Done.\n")


## Section 11 - Feature Engineering Pipeline

This section engineers features from raw data and produces ML-ready training and prediction datasets with identical feature schemas.

**Training Dataset Processing**:
1. Copies `flat_df` and filters to movies with `opening_weekend_gross_usd` (excludes movies without box office data)
2. Coerces numeric columns to float: `budget_usd`, `runtime_minutes`
3. Converts `date` column to datetime
4. **Creates `has_budget` binary flag** (1 if TMDB had budget data > 0, 0 if missing) — captured BEFORE imputation
5. Drops rows missing `runtime_minutes`, `date`, or `opening_weekend_gross_usd` (budget is NOT dropped, will be imputed)
6. **Budget Imputation (Studio + Genre Hierarchy)**: For missing/zero budgets:
   - First tries median of (studio_type, genre) combination
   - Falls back to median of studio_type only
   - Final fallback to overall dataset median
   - This ensures major studio films get ~$100M+ imputed budgets while indie films get ~$2-5M
7. Creates `release_year` from `year` column
8. Adds seasonal flags using `add_season_flags()`: `is_summer`, `is_holiday`, `is_q1_dump_window`, `is_spring_dead_zone`, `is_awards_season`
9. Creates `log_budget` using log1p transformation (now always >0 due to imputation)
10. Creates `is_collection` binary flag (1 if part of franchise)
11. Creates `is_major_studio` using `is_major_studio_from_text()`
12. Creates `is_english` binary flag (1 if `original_language` == "en")
13. **Computes `collection_past_open_avg`** using `compute_collection_past_open_avg()`: For each franchise film, calculates the mean opening weekend of all prior films in that collection (prevents data leakage by using only strictly earlier releases)
14. Ensures star-power columns exist (`director_has_prior`, `director_prior_count`, `lead_actor_has_prior`, `lead_actor_prior_count`, and conditionally `director_recent_opening`, `lead_actor_recent_opening`)
15. When `FETCH_RECENT_OPENING_FEATURES=True`, applies `enforce_star_power_validations()`; when False, sets recent-opening columns to NaN
16. Normalizes `rating_code` to standard US MPA categories (G, PG, PG-13, R, NC-17, UNRATED, OTHER) via `normalize_rating()`
17. One-hot encodes normalized ratings (creates `rating_*` columns)
18. One-hot encodes genres (creates `genre_*` columns for primary or secondary genre matches)
19. Creates `opening_weekend` target column from `opening_weekend_gross_usd`
20. Selects final feature columns: base_cols + rating one-hot + genre one-hot + studio/language/franchise/star-power features + `opening_weekend`
21. Enforces numeric types on appropriate columns
22. Validates with `validate_feature_frame(is_training=True)`
23. Saves to `preprocessed_movies_all.csv`

**Avatar 3 Prediction Dataset Processing**:
1. If `raw_avatar3_df` is not empty, extracts single row
2. Converts Avatar 3 `release_date_raw` to datetime, extracts year
3. Creates single-row dataframe with same schema as training data
4. **Creates `has_budget` flag** (1 if budget known > 0, 0 if missing)
5. **Applies same budget imputation logic** if Avatar 3 budget is missing (uses training set studio-type medians)
6. Applies identical transformations: season flags, `log_budget`, `is_collection`, `is_major_studio`, `is_english`
7. **Computes `collection_past_open_avg`** for Avatar 3 using mean of all prior Avatar films in training set (no data leakage)
8. Handles star-power columns (uses computed values from `append_star_power_columns()`, or NaN if toggle is off)
9. One-hot encodes rating and genres using same encoding scheme as training set
10. Sets `opening_weekend` to NaN (prediction target)
11. Selects same `final_cols` as training set
12. Enforces numeric types on same columns
13. Validates with `validate_feature_frame(is_training=False)`
14. Saves to `preprocessed_avatar3_only.csv`

**Returns**: Tuple of (training_features, avatar_features). Avatar features is None if `raw_avatar3_df` is empty.

**Critical Design**: 
- Avatar 3 uses training set statistics (budget medians, collection averages) to prevent data leakage
- Feature engineering is identical for both datasets to ensure model compatibility
- `has_budget` flag allows model to learn that missing budgets are predictive (often indie/limited releases)
- Budget imputation is studio-informed, so major studios get realistic ~$100M+ imputed budgets
- `collection_past_open_avg` provides powerful franchise history signal for sequel predictions
- Recent-opening features are gated by `FETCH_RECENT_OPENING_FEATURES` toggle to avoid expensive API calls

**Summary Statistics**: After completion, prints comprehensive feature engineering metrics including:
- Training dataset size and total feature count
- Feature breakdown by category (seasonal, rating one-hot, genre one-hot, economic, studio/language, data quality, franchise, star power)
- Target variable statistics (min/median/mean/max/std for `opening_weekend`)
- Key feature distributions showing percentages of major studios, known budgets, franchises, English-language films, seasonal releases
- Avatar 3 prediction dataset confirmation

In [27]:
def build_feature_datasets(flat_df: pd.DataFrame, raw_avatar3_df: pd.DataFrame) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    df = flat_df.copy()
    df = df[df["opening_weekend_gross_usd"].notna()].copy()

    df["budget_usd"] = pd.to_numeric(df["budget_usd"], errors="coerce").astype(float)
    df["runtime_minutes"] = pd.to_numeric(df["runtime_minutes"], errors="coerce").astype(float)

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    
    # Create has_budget flag BEFORE any imputation or dropping
    df["has_budget"] = ((df["budget_usd"].notna()) & (df["budget_usd"] > 0)).astype(int)
    
    # Now drop only rows missing runtime, date, and opening weekend (NOT budget)
    df = df.dropna(subset=["runtime_minutes", "date", "opening_weekend_gross_usd"])

    # === BUDGET IMPUTATION: Studio + Genre Hierarchy ===
    # Identify which rows need imputation
    needs_imputation = (df["budget_usd"].isna()) | (df["budget_usd"] == 0)
    
    if needs_imputation.any():
        # First, identify major studios
        df["is_major_studio"] = df["production_companies"].apply(is_major_studio_from_text)
        
        # Imputation hierarchy: (studio_type, genre) -> studio_type -> overall
        for is_major in [True, False]:
            studio_mask = df["is_major_studio"] == is_major
            
            for genre in df["primary_genre"].dropna().unique():
                genre_mask = studio_mask & (df["primary_genre"] == genre)
                
                # Calculate median for this (studio_type, genre) combo
                combo_median = df.loc[
                    genre_mask & (df["budget_usd"] > 0), 
                    "budget_usd"
                ].median()
                
                # Fill missing in this combo
                impute_mask = genre_mask & needs_imputation
                if impute_mask.any() and combo_median > 0:
                    df.loc[impute_mask, "budget_usd"] = combo_median
        
        # Fallback: Impute by studio type only (for genres with no data)
        for is_major in [True, False]:
            studio_mask = df["is_major_studio"] == is_major
            studio_median = df.loc[
                studio_mask & (df["budget_usd"] > 0), 
                "budget_usd"
            ].median()
            
            impute_mask = studio_mask & ((df["budget_usd"].isna()) | (df["budget_usd"] == 0))
            if impute_mask.any() and studio_median > 0:
                df.loc[impute_mask, "budget_usd"] = studio_median
        
        # Final fallback: Overall median
        overall_median = df[df["budget_usd"] > 0]["budget_usd"].median()
        df["budget_usd"] = df["budget_usd"].fillna(overall_median)
        df.loc[df["budget_usd"] == 0, "budget_usd"] = overall_median
    
    df["release_year"] = df["year"].astype("Int64")

    df = add_season_flags(df, "date")

    df["log_budget"] = np.log1p(df["budget_usd"]).astype(float)

    df["is_collection"] = df["collection_name"].notna().astype(int)

    # is_major_studio already computed during imputation, but recompute here to ensure consistency
    df["is_major_studio"] = df["production_companies"].apply(is_major_studio_from_text)

    df["is_english"] = (df["original_language"] == "en").astype(int)

    df = compute_collection_past_open_avg(df)
    df["collection_past_open_avg"] = pd.to_numeric(
        df["collection_past_open_avg"], errors="coerce"
    ).fillna(0.0).astype(float)

    # Ensure star-power columns exist even if upstream computation failed
    for col in [
        "director_has_prior",
        "director_prior_count",
        "director_recent_opening",
        "lead_actor_has_prior",
        "lead_actor_prior_count",
        "lead_actor_recent_opening",
    ]:
        if col not in df.columns:
            df[col] = np.nan

    # GUARD: Only validate and keep recent-opening columns if FETCH_RECENT_OPENING_FEATURES is True
    if FETCH_RECENT_OPENING_FEATURES:
        df = enforce_star_power_validations(df)
    else:
        # When toggle is OFF, drop the expensive recent-opening columns
        df["director_recent_opening"] = np.nan
        df["lead_actor_recent_opening"] = np.nan

    df["director_has_prior"] = df["director_has_prior"].fillna(0).astype(int)
    df["lead_actor_has_prior"] = df["lead_actor_has_prior"].fillna(0).astype(int)

    for cnt_col in ["director_prior_count", "lead_actor_prior_count"]:
        df[cnt_col] = pd.to_numeric(df[cnt_col], errors="coerce")
    
    # GUARD: Only include recent-opening columns in final output if toggle is True
    for rec_col in ["director_recent_opening", "lead_actor_recent_opening"]:
        df[rec_col] = pd.to_numeric(df[rec_col], errors="coerce")

    # Normalize ratings to standard US MPA categories (reduces 104 variants to 7 categories)
    df["rating_normalized"] = df["rating_code"].apply(normalize_rating)

    # Define standard rating order (these are the only rating columns we'll create)
    standard_ratings = ["G", "PG", "PG-13", "R", "NC-17", "UNRATED", "OTHER"]
    rating_cols: List[str] = []
    for code in standard_ratings:
        col = f"rating_{code.replace('-', '_')}"
        df[col] = (df["rating_normalized"] == code).astype(int)
        rating_cols.append(col)

    primary = df["primary_genre"].fillna("").astype(str)
    secondary = df["secondary_genre"].fillna("").astype(str)

    all_genres = sorted(
        set([g for g in primary.unique() if g] + [g for g in secondary.unique() if g])
    )

    genre_cols: List[str] = []
    for g in all_genres:
        safe = re.sub(r"[^A-Za-z0-9]+", "_", g.strip().lower())
        col = f"genre_{safe}"
        df[col] = ((primary == g) | (secondary == g)).astype(int)
        genre_cols.append(col)

    df["opening_weekend"] = pd.to_numeric(
        df["opening_weekend_gross_usd"], errors="coerce"
    ).astype(float)

    base_cols = [
        "title",
        "release_year",
        "is_summer",
        "is_holiday",
        "is_q1_dump_window",
        "is_spring_dead_zone",
        "is_awards_season",
        "log_budget",
        "has_budget",
        "runtime_minutes",
    ]
    
    # GUARD: Conditionally include recent-opening columns in middle_cols
    middle_cols = rating_cols + genre_cols + [
        "is_major_studio",
        "is_english",
        "is_collection",
        "collection_past_open_avg",
        "director_has_prior",
        "director_prior_count",
        "lead_actor_has_prior",
        "lead_actor_prior_count",
    ]
    
    # Only add recent-opening columns if toggle is ON
    if FETCH_RECENT_OPENING_FEATURES:
        middle_cols.extend([
            "director_recent_opening",
            "lead_actor_recent_opening",
        ])
    
    final_cols = base_cols + middle_cols + ["opening_weekend"]

    features_training = df[final_cols].copy()

    numeric_train_cols = [
        "log_budget",
        "runtime_minutes",
        "has_budget",
        "collection_past_open_avg",
        "director_prior_count",
        "lead_actor_prior_count",
        "opening_weekend",
    ]
    
    # Only include recent-opening numeric columns if toggle is ON
    if FETCH_RECENT_OPENING_FEATURES:
        numeric_train_cols.extend([
            "director_recent_opening",
            "lead_actor_recent_opening",
        ])
    
    for col in numeric_train_cols:
        features_training[col] = pd.to_numeric(features_training[col], errors="coerce").astype(float)

    validate_feature_frame(features_training, is_training=True)

    train_out_path = os.path.join(DATA_DIR, "preprocessed_movies_all.csv")
    features_training.to_csv(train_out_path, index=False)
    print(f" Saved engineered training features → {train_out_path} (rows: {len(features_training)})")

    avatar_feat: Optional[pd.DataFrame] = None
    if not raw_avatar3_df.empty:
        row = raw_avatar3_df.iloc[0].copy()

        avatar_date = pd.to_datetime(row.get("release_date_raw"), errors="coerce")
        avatar_year = avatar_date.year if pd.notna(avatar_date) else None

        a_df = pd.DataFrame([{
            "title": row.get("title"),
            "date": avatar_date,
            "budget_usd": pd.to_numeric(row.get("budget_usd"), errors="coerce"),
            "runtime_minutes": pd.to_numeric(row.get("runtime_minutes"), errors="coerce"),
            "collection_name": row.get("collection_name"),
            "primary_genre": row.get("primary_genre"),
            "secondary_genre": row.get("secondary_genre"),
            "rating_code": row.get("mpa_rating_raw") or "UNKNOWN",
            "production_companies": row.get("production_companies"),
            "original_language": row.get("original_language"),
            "director_has_prior": row.get("director_has_prior"),
            "director_prior_count": row.get("director_prior_count"),
            "director_recent_opening": row.get("director_recent_opening") if FETCH_RECENT_OPENING_FEATURES else np.nan,
            "lead_actor_has_prior": row.get("lead_actor_has_prior"),
            "lead_actor_prior_count": row.get("lead_actor_prior_count"),
            "lead_actor_recent_opening": row.get("lead_actor_recent_opening") if FETCH_RECENT_OPENING_FEATURES else np.nan,
        }])

        # Track has_budget for Avatar 3
        a_df["has_budget"] = ((a_df["budget_usd"].notna()) & (a_df["budget_usd"] > 0)).astype(int)

        a_df["budget_usd"] = a_df["budget_usd"].astype(float)
        a_df["runtime_minutes"] = a_df["runtime_minutes"].astype(float)

        # Apply same imputation logic if Avatar 3 budget is missing
        if (a_df["budget_usd"].isna().any() or (a_df["budget_usd"] == 0).any()):
            a_df["is_major_studio"] = a_df["production_companies"].apply(is_major_studio_from_text)
            
            # Use the training set's median values for imputation
            major_studio_median = df[df["is_major_studio"] & (df["budget_usd"] > 0)]["budget_usd"].median()
            indie_median = df[~df["is_major_studio"] & (df["budget_usd"] > 0)]["budget_usd"].median()
            
            if a_df["is_major_studio"].iloc[0]:
                if major_studio_median > 0:
                    a_df.loc[a_df["budget_usd"].isna() | (a_df["budget_usd"] == 0), "budget_usd"] = major_studio_median
            else:
                if indie_median > 0:
                    a_df.loc[a_df["budget_usd"].isna() | (a_df["budget_usd"] == 0), "budget_usd"] = indie_median
            
            # Final fallback
            overall_median = df[df["budget_usd"] > 0]["budget_usd"].median()
            a_df["budget_usd"] = a_df["budget_usd"].fillna(overall_median)
            a_df.loc[a_df["budget_usd"] == 0, "budget_usd"] = overall_median

        a_df["release_year"] = avatar_year

        a_df = add_season_flags(a_df, "date")
        a_df["log_budget"] = np.log1p(a_df["budget_usd"]).astype(float)
        a_df["is_collection"] = a_df["collection_name"].notna().astype(int)
        a_df["is_major_studio"] = a_df["production_companies"].apply(is_major_studio_from_text)
        a_df["is_english"] = (a_df["original_language"] == "en").astype(int)

        coll_name = row.get("collection_name")
        if coll_name and coll_name in df["collection_name"].dropna().unique():
            past = df[df["collection_name"] == coll_name]["opening_weekend_gross_usd"]
            if not past.empty:
                a_df["collection_past_open_avg"] = float(past.mean())
            else:
                a_df["collection_past_open_avg"] = 0.0
        else:
            a_df["collection_past_open_avg"] = 0.0
        a_df["collection_past_open_avg"] = a_df["collection_past_open_avg"].astype(float)

        # GUARD: Only apply validation if toggle is ON
        if FETCH_RECENT_OPENING_FEATURES:
            a_df = enforce_star_power_validations(a_df)
        else:
            # When toggle is OFF, ensure recent-opening columns are NaN
            a_df["director_recent_opening"] = np.nan
            a_df["lead_actor_recent_opening"] = np.nan

        # Normalize Avatar 3 rating using same logic as training data
        a_df["rating_normalized"] = a_df["rating_code"].apply(normalize_rating)
        
        for code, col in zip(standard_ratings, rating_cols):
            a_df[col] = (a_df["rating_normalized"] == code).astype(int)

        a_primary = a_df["primary_genre"].fillna("").astype(str)
        a_secondary = a_df["secondary_genre"].fillna("").astype(str)
        for g, col in zip(all_genres, genre_cols):
            a_df[col] = ((a_primary == g) | (a_secondary == g)).astype(int)

        a_df["opening_weekend"] = np.nan

        avatar_feat = a_df[final_cols].copy()

        numeric_avatar_cols = [
            "log_budget",
            "runtime_minutes",
            "has_budget",
            "collection_past_open_avg",
            "director_prior_count",
            "lead_actor_prior_count",
            "opening_weekend",
        ]
        
        # Only include recent-opening numeric columns if toggle is ON
        if FETCH_RECENT_OPENING_FEATURES:
            numeric_avatar_cols.extend([
                "director_recent_opening",
                "lead_actor_recent_opening",
            ])
        
        for col in numeric_avatar_cols:
            avatar_feat[col] = pd.to_numeric(avatar_feat[col], errors="coerce").astype(float)

        validate_feature_frame(avatar_feat, is_training=False)

        avatar_out_path = os.path.join(DATA_DIR, "preprocessed_avatar3_only.csv")
        avatar_feat.to_csv(avatar_out_path, index=False)
        print(f" Saved engineered Avatar 3 features → {avatar_out_path}")
    
    # === FEATURE ENGINEERING SUMMARY ===
    print("\n" + "="*70)
    print("FEATURE ENGINEERING SUMMARY")
    print("="*70)
    print(f"FETCH_RECENT_OPENING_FEATURES: {FETCH_RECENT_OPENING_FEATURES}")
    print(f"Training dataset: {len(features_training):,} movies")
    print(f"Features: {len(final_cols)-1} (excluding 'opening_weekend' target)")
    
    # Feature type breakdown
    rating_count = len(rating_cols)
    genre_count = len(genre_cols)
    seasonal_count = 5
    star_power_count = 4 if not FETCH_RECENT_OPENING_FEATURES else 6  # has_prior, prior_count, [recent_opening]
    print(f"\nFeature categories:")
    print(f"  - Seasonal flags: {seasonal_count}")
    print(f"  - Rating one-hot: {rating_count}")
    print(f"  - Genre one-hot: {genre_count}")
    print(f"  - Economic: 2 (log_budget, runtime_minutes)")
    print(f"  - Studio/language: 2 (is_major_studio, is_english)")
    print(f"  - Data quality: 1 (has_budget)")
    print(f"  - Franchise: 2 (is_collection, collection_past_open_avg)")
    print(f"  - Star power: {star_power_count} (director/lead history)")
    if FETCH_RECENT_OPENING_FEATURES:
        print(f"    └─ NOTE: Including director_recent_opening & lead_actor_recent_opening")
    else:
        print(f"    └─ NOTE: Excluding director_recent_opening & lead_actor_recent_opening (toggle=False)")
    
    # Target variable statistics
    print(f"\nTarget variable (opening_weekend):")
    print(f"  - Min: ${features_training['opening_weekend'].min():,.0f}")
    print(f"  - Median: ${features_training['opening_weekend'].median():,.0f}")
    print(f"  - Mean: ${features_training['opening_weekend'].mean():,.0f}")
    print(f"  - Max: ${features_training['opening_weekend'].max():,.0f}")
    print(f"  - Std Dev: ${features_training['opening_weekend'].std():,.0f}")
    
    # Distribution of key binary features
    print(f"\nKey feature distributions:")
    print(f"  - Major studio releases: {features_training['is_major_studio'].sum()} ({features_training['is_major_studio'].mean()*100:.1f}%)")
    print(f"  - Known budget in TMDB: {features_training['has_budget'].sum()} ({features_training['has_budget'].mean()*100:.1f}%)")
    print(f"  - Franchise films: {features_training['is_collection'].sum()} ({features_training['is_collection'].mean()*100:.1f}%)")
    print(f"  - English language: {features_training['is_english'].sum()} ({features_training['is_english'].mean()*100:.1f}%)")
    print(f"  - Summer releases: {features_training['is_summer'].sum()} ({features_training['is_summer'].mean()*100:.1f}%)")
    print(f"  - Holiday releases: {features_training['is_holiday'].sum()} ({features_training['is_holiday'].mean()*100:.1f}%)")
    
    if avatar_feat is not None:
        print(f"\nAvatar 3 prediction dataset ready (1 row, {len(final_cols)-1} features)")
    
    print("="*70 + "\n")
    
    return features_training, avatar_feat


## Section 12 - Main

This section orchestrates the entire pipeline by calling all staging, warehouse, and analytics functions in sequence.

**Execution Flow**:
1. Validates TMDB API key is configured
2. **Staging Layer**:
   - Builds TMDB staging table (`stg_tmdb.csv`)
   - Builds opening weekend staging table (`stg_opening.csv`)
3. **Warehouse Layer**:
   - Builds date dimension and date_key lookup map
   - Builds rating dimension and rating_key lookup map
   - Builds studio dimension, bridge table, and studio_key lookup map
   - Builds movie dimension and movie_key lookup map
   - Builds fact_opening_weekend table
4. **Analytics Layer**:
   - Builds flattened raw dataset (`raw_movies_all.csv`)
   - Builds Avatar 3 raw snapshot (`raw_avatar3_only.csv`)
   - Builds engineered feature datasets (`preprocessed_movies_all.csv`, `preprocessed_avatar3_only.csv`)
5. **Cleanup** (optional):
   - If DELETE_CACHE_ON_FINISH is True, removes HTML cache directory
6. Prints completion message with output directory and file list

**Entry Point**: Script executes `main()` when run directly (`if __name__ == "__main__"`).

All outputs are saved to DATA_DIR. Pipeline must run sequentially as each stage depends on outputs from previous stages.

In [28]:
def main() -> None:
    _require_tmdb_key()

    tmdb_stg = build_tmdb_staging()
    open_stg = build_opening_staging(tmdb_stg)

    dim_date, date_key_map = build_dim_date(tmdb_stg)
    dim_rating, rating_key_map = build_dim_rating(tmdb_stg)
    dim_studio, bridge_movie_studio, _ = build_dim_studio_and_bridge(tmdb_stg)
    dim_movie, movie_key_map = build_dim_movie(tmdb_stg, date_key_map, rating_key_map)

    fact_opening = build_fact_opening(open_stg, movie_key_map)

    flat_df = build_flat_raw_dataset(
        dim_movie=dim_movie,
        dim_date=dim_date,
        dim_rating=dim_rating,
        dim_studio=dim_studio,
        bridge_movie_studio=bridge_movie_studio,
        fact_opening=fact_opening,
        open_stg=open_stg,
    )

    raw_avatar3_df = build_raw_avatar3_only(open_stg=open_stg)

    build_feature_datasets(flat_df, raw_avatar3_df)

    if DELETE_CACHE_ON_FINISH:
        import shutil
        shutil.rmtree(CACHE_DIR, ignore_errors=True)
        print(" HTML cache cleared.")

    print("=== DONE ===")
    print("Star schema CSVs + flat raw CSVs + engineered feature CSVs created in:", DATA_DIR)
    print("Files:", os.listdir(DATA_DIR))


if __name__ == "__main__":
    main()

Discovering TMDB titles from 2007 to 2025 ...


Years: 100%|██████████| 19/19 [06:23<00:00, 20.18s/year]


[TMDB] Unique discovered movies: 14813
[TMDB] Remaining to fetch this run: 14813


Fetching TMDB movie details: 100%|██████████| 14813/14813 [12:21<00:00, 19.97movie/s]



TMDB STAGING SUMMARY
Movies fetched this run: 14,813
Total movies in staging: 14,290
Excluded 523 short film(s) (<60 min)
Date range: 2007-01-01 to 2025-12-25

Missing data:
  - IMDb IDs: 5 (0.0%)
  - Budget: 0 (0.0%)
  - Runtime: 0 (0.0%)

Budget statistics (n=5736):
  - Min: $1
  - Median: $12,000,000
  - Mean: $29,878,793
  - Max: $489,900,000

Runtime statistics (n=14290):
  - Min: 60 min
  - Median: 100 min
  - Max: 585 min

Top 5 genres:
  - Drama: 3188 (22.3%)
  - Comedy: 2879 (20.1%)
  - Action: 1692 (11.8%)
  - Horror: 1399 (9.8%)
  - Thriller: 859 (6.0%)

Scraping DOMESTIC opening-weekend box office (BOM -> The Numbers)...


...

[STAR-POWER] Precomputing recent opening weekends for unique people...


Lead actors (recent opening): 100%|██████████| 1/1 [00:02<00:00,  2.67s/person]


[STAR-POWER] Merging precomputed metrics into dataframe...
[STAR-POWER] Feature computation complete.
 Saved Avatar 3 raw TMDB snapshot → star_schema_output\raw_avatar3_only.csv


Processing franchise collections: 100%|██████████| 1032/1032 [37:51<00:00,  2.20s/collection] 



[VALIDATION] Feature frame summary:
  rows: 7725
  ✅ No missing values in required core columns.
  ✅ Column is_summer is binary (0/1).
  ✅ Column is_holiday is binary (0/1).
  ✅ Column is_q1_dump_window is binary (0/1).
  ✅ Column is_spring_dead_zone is binary (0/1).
  ✅ Column is_awards_season is binary (0/1).
  ✅ Column is_major_studio is binary (0/1).
  ✅ Column is_english is binary (0/1).
  ✅ Column is_collection is binary (0/1).
  ✅ Column director_has_prior is binary (0/1).
  ✅ Column lead_actor_has_prior is binary (0/1).
  ✅ Training targets (opening_weekend) look valid.
[VALIDATION] Done.

 Saved engineered training features → star_schema_output\preprocessed_movies_all.csv (rows: 7725)

[VALIDATION] Feature frame summary:
  rows: 1
  ✅ No missing values in required core columns.
  ✅ Column is_summer is binary (0/1).
  ✅ Column is_holiday is binary (0/1).
  ✅ Column is_q1_dump_window is binary (0/1).
  ✅ Column is_spring_dead_zone is binary (0/1).
  ✅ Column is_awards_season is